<a href="https://colab.research.google.com/github/AnnaPaulaFigueiredo/data_master/blob/main/01_DM_CASE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [ ]:
from google.colab import drive
import warnings
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import shutil
import warnings
from pyspark.sql import functions as F
import shutil
from pyspark.sql import DataFrame

# Desativa os avisos em uma linha própria
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option('display.max_columns', 500)
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import NumericType, DateType, TimestampType, DoubleType, FloatType
import humanize
from pyspark.sql.functions import to_date, col, concat, lit
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType, DateType, TimestampType, StringType
if 'spark' not in locals():
    spark = SparkSession.builder.appName("DM_CASE").getOrCreate()
import pandas as pd
import shutil
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, to_date
import shutil
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import shutil
from pyspark.sql import functions as F
from pyspark.sql import Window
pd.set_option('display.max_columns', None)

ids = [
    "++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=",
    "+++EI4HgyhgcJHIPXk/VRP7bt17+2joG39T6oEfJ+tc=",
    "+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=",
    "+++FOrTS7ab3tIgIh8eWwX4FqRv8w/FoiOuyXsFvphY=",
    "+++4vcS9aMH7KWdfh5git6nA5fC5jjisd5H/NcM++WM=",
    "++/TR7WI15q2ZCtOXmoap7jR+kEhbMVE5swOqsfqpqI=",
    "+++4vcS9aMH7KWdfh5git6nA5fC5jjisd5H/NcM++WM="
]

# Global vars

In [ ]:
IDENTIFY = "msno"
TIME_COL = "safra"

output_analysis_path = "/content/drive/MyDrive/SANTANDER/merged_ids_analysis.parquet"
dfs_members = spark.read.parquet("/content/drive/MyDrive/SANTANDER/members.parquet")
dfs_transactions = spark.read.parquet("/content/drive/MyDrive/SANTANDER/transactions.parquet")
dfs_logs = spark.read.parquet("/content/drive/MyDrive/SANTANDER/user_logs.parquet")

# Functions

## Data Quality

In [ ]:
def void_input(df, col, value):
    return df.withColumn(
        col,
        F.when(F.col(col).isNull(), value).otherwise(F.col(col))
    )

def is_numeric_string(sample):
    try:
        return all(float(x) or True for x in sample if x is not None)
    except:
        return False

def analyse_columns(df_spark):
    resultado = []
    total_linhas = df_spark.count()

    for campo in df_spark.schema.fields:
        nome = campo.name
        tipo = campo.dataType

        unicos_sample = (
            df_spark.select(nome)
            .distinct()
            .limit(10)
            .toPandas()[nome]
            .tolist()
        )

        n_valores_unicos = df_spark.select(nome).distinct().count()

        if isinstance(tipo, NumericType):
            tipo_base = "Numérico"

        elif isinstance(tipo, (DateType, TimestampType)):
            tipo_base = "Data"

        elif isinstance(tipo, StringType):
            if is_numeric_string(unicos_sample):
                tipo_base = "Numérico (string)"
            else:
                tipo_base = "Categórico"

        else:
            tipo_base = "Outro"


        if isinstance(tipo, NumericType):
            nulos = df_spark.filter(
                F.col(nome).isNull() | F.isnan(F.col(nome))
            ).count()
        else:
            nulos = df_spark.filter(F.col(nome).isNull()).count()

        duplicados = (
            df_spark.groupBy(nome)
            .count()
            .filter(F.col("count") > 1)
            .count()
        )


        eh_binario = n_valores_unicos == 2


        eh_ordinal = False

        if tipo_base.startswith("Numérico") and 2 < n_valores_unicos <= 10:
            eh_ordinal = True

        elif tipo_base == "Categórico":
            if n_valores_unicos <= 6:
                eh_ordinal = True

        if tipo_base.startswith("Numérico") or tipo_base == "Data":
            stats = df_spark.select(
                F.min(F.col(nome)).alias("minimo"),
                F.max(F.col(nome)).alias("maximo")
            ).first()

            minimo = stats["minimo"]
            maximo = stats["maximo"]
        else:
            minimo = None
            maximo = None

        resultado.append({
            "coluna": nome,
            "tipo_base": tipo_base,
            "ordinal": eh_ordinal,
            "binário": eh_binario,
            "mínimo": minimo,
            "máximo": maximo,
            "nulos": nulos,
            "duplicados": duplicados,
            "n° valores únicos": n_valores_unicos,
            "valores únicos (sample)": unicos_sample[:5] + (["..."] if len(unicos_sample) > 5 else [])
        })

    return pd.DataFrame(resultado)


def cast_columns(
    df,
    cols_int=None,
    cols_float=None,
    cols_string=None,
    cols_date=None,
    date_format="yyyy-MM-dd"
):
    """
    Converte colunas de um DataFrame para tipos específicos.

    Parâmetros:
    - df: DataFrame Spark
    - cols_int: lista de colunas para int
    - cols_float: lista de colunas para float/double
    - cols_string: lista de colunas para string
    - cols_date: lista de colunas para date
    - date_format: formato das datas (default: yyyy-MM-dd)

    Retorna:
    - DataFrame com casts aplicados
    """

    cols_int = cols_int or []
    cols_float = cols_float or []
    cols_string = cols_string or []
    cols_date = cols_date or []

    df_out = df

    # Inteiro
    for c in cols_int:
        df_out = df_out.withColumn(c, col(c).cast("int"))

    # Float / Double
    for c in cols_float:
        df_out = df_out.withColumn(c, col(c).cast("double"))

    # String
    for c in cols_string:
        df_out = df_out.withColumn(c, col(c).cast("string"))

    # Date
    for c in cols_date:
        df_out = df_out.withColumn(c, to_date(col(c), date_format))

    return df_out

## Examples

In [ ]:
%%time
dfs_members = spark.read.parquet("/content/drive/MyDrive/SANTANDER/members.parquet")
dfs_members.filter((F.col('msno') == ids[0])).orderBy(F.col("safra")).show(truncate=False)
df_stats = analyse_columns(dfs_members)
df_stats

+--------------------------------------------+------+----------------------+----+---+------+--------------+--------+
|msno                                        |safra |registration_init_time|city|bd |gender|registered_via|is_ativo|
+--------------------------------------------+------+----------------------+----+---+------+--------------+--------+
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|201601|20140421              |13  |39 |male  |3             |1       |
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|201602|20140421              |13  |39 |male  |3             |1       |
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|201603|20140421              |13  |39 |male  |3             |1       |
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|201604|20140421              |13  |39 |male  |3             |1       |
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|201605|20140421              |13  |39 |male  |3             |1       |
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|201606|20140421   

,coluna,tipo_base,ordinal,binário,mínimo,máximo,nulos,duplicados,n° valores únicos,valores únicos (sample)
0,msno,Categórico,False,False,None,None,0,6114345,6287789,"[++7jKYbuIJPXry8Oh1NcEh9fCsqcQgUaaxXsgG15kMg=,..."
1,safra,Numérico (string),False,False,201601,201612,0,12,12,"[201607, 201611, 201605, 201602, 201606, ...]"
2,registration_init_time,Numérico (string),False,False,20040326,20161231,0,4663,4663,"[20150729, 20120909, 20120322, 20160429, 20141..."
3,city,Numérico (string),False,False,1,9,0,21,21,"[7, 15, 11, 3, 8, ...]"
4,bd,Numérico (string),False,False,-10,994,0,381,385,"[944, 51, 7, -489, 124, ...]"
5,gender,Categórico,True,False,None,None,38210177,3,3,"[female, male, None]"
6,registered_via,Numérico (string),False,False,-1,9,0,17,17,"[7, 11, 3, 8, 16, ...]"
7,is_ativo,Numérico,False,True,0,1,0,2,2,"[1, 0]"


In [ ]:
%%time
dfs_transactions = spark.read.parquet("/content/drive/MyDrive/SANTANDER/transactions.parquet")
dfs_transactions.filter((F.col('msno') == ids[0])).orderBy(F.col("safra")).show(truncate=False)
df_stats = analyse_columns(dfs_transactions)
df_stats

+--------------------------------------------+-----------------+-----------------+---------------+------------------+-------------+----------------+----------------------+---------+------+
|msno                                        |payment_method_id|payment_plan_days|plan_list_price|actual_amount_paid|is_auto_renew|transaction_date|membership_expire_date|is_cancel|safra |
+--------------------------------------------+-----------------+-----------------+---------------+------------------+-------------+----------------+----------------------+---------+------+
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|14               |0                |0              |149               |1            |20150331        |20150430              |0        |201503|
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|14               |0                |0              |149               |1            |20150630        |20150731              |0        |201506|
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|14       

,coluna,tipo_base,ordinal,binário,mínimo,máximo,nulos,duplicados,n° valores únicos,valores únicos (sample)
0,msno,Categórico,False,False,None,None,0,1697963,2363626,"[++4RuqBw0Ss6bQU4oMxaRlbBPoWzoEiIZaxPM04Y4+U=,..."
1,payment_method_id,Numérico (string),False,False,1,8,0,40,40,"[7, 15, 11, 29, 3, ...]"
2,payment_plan_days,Numérico (string),False,False,0,99,0,34,37,"[7, 15, 200, 3, 30, ...]"
3,plan_list_price,Numérico (string),False,False,0,99,0,49,51,"[447, 124, 15, 894, 1788, ...]"
4,actual_amount_paid,Numérico (string),False,False,0,99,0,51,57,"[447, 124, 15, 894, 1788, ...]"
5,is_auto_renew,Numérico (string),False,True,0,1,0,2,2,"[0, 1]"
6,transaction_date,Numérico (string),False,False,20150101,20170228,0,790,790,"[20160615, 20160820, 20161119, 20160825, 20150..."
7,membership_expire_date,Numérico (string),False,False,19700101,20170331,0,1473,1534,"[20160615, 20161119, 20160429, 20160820, 20160..."
8,is_cancel,Numérico (string),False,True,0,1,0,2,2,"[0, 1]"
9,safra,Numérico,False,False,201501,201702,0,26,26,"[201505, 201701, 201702, 201501, 201509, ...]"


In [ ]:
%%time
dfs_log = spark.read.parquet("/content/drive/MyDrive/SANTANDER/user_logs.parquet")
dfs_log.filter((F.col('msno') == ids[0])).orderBy(F.col("safra")).show(truncate=False)
df_stats = analyse_columns(dfs_log)
df_stats

+--------------------------------------------+------+------+------+------+-------+-------+-------+------------------+
|msno                                        |safra |num_25|num_50|num_75|num_985|num_100|num_unq|total_secs        |
+--------------------------------------------+------+------+------+------+-------+-------+-------+------------------+
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|201501|7.0   |1.0   |3.0   |2.0    |63.0   |60.0   |16674.686         |
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|201502|32.0  |10.0  |5.0   |8.0    |487.0  |328.0  |116105.431        |
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|201503|6.0   |2.0   |2.0   |1.0    |139.0  |138.0  |34775.509         |
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|201504|5.0   |4.0   |1.0   |0.0    |169.0  |168.0  |43534.784         |
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|201505|3.0   |0.0   |2.0   |0.0    |25.0   |29.0   |6340.997          |
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|201506|0.0

,coluna,tipo_base,ordinal,binário,mínimo,máximo,nulos,duplicados,n° valores únicos,valores únicos (sample)
0,msno,Categórico,False,False,NaN,NaN,0,2293336,5234111,"[21Ld7rZnJm9Sa6+Lc7JVsvAbdX2F8g7dIe8vIqs4B6s=,..."
1,safra,Numérico,False,False,2.015010e+05,2.017020e+05,0,26,26,"[201505, 201701, 201702, 201501, 201509, ...]"
2,num_25,Numérico,False,False,0.000000e+00,1.118640e+05,0,3816,4891,"[596.0, 299.0, 305.0, 692.0, 1051.0, ...]"
3,num_50,Numérico,False,False,0.000000e+00,8.875000e+03,0,1233,1587,"[299.0, 305.0, 558.0, 496.0, 769.0, ...]"
4,num_75,Numérico,False,False,0.000000e+00,3.485000e+03,0,746,986,"[299.0, 305.0, 596.0, 170.0, 147.0, ...]"
5,num_985,Numérico,False,False,0.000000e+00,3.769800e+04,0,1740,2509,"[299.0, 596.0, 305.0, 558.0, 769.0, ...]"
6,num_100,Numérico,False,False,0.000000e+00,1.967410e+05,0,10573,13047,"[305.0, 299.0, 692.0, 596.0, 2815.0, ...]"
7,num_unq,Numérico,False,False,1.000000e+00,3.270600e+04,0,7450,8593,"[934.0, 305.0, 596.0, 496.0, 558.0, ...]"
8,total_secs,Numérico,False,False,-2.398077e+17,9.223372e+15,0,1395426,24025199,"[219986.77300000002, 121822.54300000002, 80922..."


# Check joins

O objetivo é manter nas bases, só os clientes que tem dados nas três tabelas, por completo.

Para otimizar os cálculos a priori.

Além de idêntificar os clientes com histórico contúnuo de no mínimo 3 meses, para podermos identificar um padrão de comportamento.

## Ids Analysis

In [ ]:
%%time
def adicionar_meses(ano, mes, delta):
    """
    Adiciona ou subtrai meses de uma combinação ano/mês.

    Exemplo:
        adicionar_meses(2016, 1, -3)
        -> (2015, 10)
    """

    indice = ano * 12 + (mes - 1) + delta

    novo_ano = indice // 12
    novo_mes = indice % 12 + 1

    return novo_ano, novo_mes


# ============================================================
# ANALISAR BASES E IDENTIFICAR PÚBLICO-ALVO
# ============================================================

def analisar_bases(
    df1: DataFrame,
    df2: DataFrame,
    df3: DataFrame,
    nome1="base1",
    nome2="base2",
    nome3="base3",
    safra_modelagem_inicial="201601",
    meses_historico=3
):
    """
    Identifica o público-alvo por cliente e safra.

    Um cliente é elegível em determinada safra quando:

        1. Está presente nas três bases naquela safra;
        2. Possui os N meses imediatamente anteriores em TRANSACTIONS;
        3. Possui os N meses imediatamente anteriores em LOGS.

    Exemplo com 3 meses de histórico:

        Safra 201601
            histórico: 201510, 201511, 201512

        Safra 201602
            histórico: 201511, 201512, 201601

        Safra 201603
            histórico: 201512, 201601, 201602

    IMPORTANTE:
        A janela histórica é móvel e calculada individualmente
        para cada safra de modelagem.
    """

    print("=" * 70)
    print("📊 ANÁLISE DE INTERSECÇÃO DAS BASES")
    print("=" * 70)

    # ========================================================
    # 1. DOCUMENTAÇÃO DA PRIMEIRA SAFRA
    # ========================================================

    ano_modelagem = int(safra_modelagem_inicial[:4])
    mes_modelagem = int(safra_modelagem_inicial[4:])

    safras_historico_inicial = []

    for i in range(meses_historico, 0, -1):

        ano, mes = adicionar_meses(
            ano_modelagem,
            mes_modelagem,
            -i
        )

        safras_historico_inicial.append(
            f"{ano:04d}{mes:02d}"
        )

    print(
        f"📅 Primeira safra de modelagem : "
        f"{safra_modelagem_inicial}"
    )

    print(
        f"📅 Histórico mínimo necessário : "
        f"{meses_historico} meses"
    )

    print(
        f"📅 Para {safra_modelagem_inicial}, "
        f"histórico: {safras_historico_inicial}"
    )

    # ========================================================
    # 2. PADRONIZA SAFRA COMO DATE
    # ========================================================

    df1_base = (
        df1
        .select(
            "msno",
            F.to_date(
                F.col("safra").cast("string"),
                "yyyyMM"
            ).alias("safra_date")
        )
        .filter(F.col("safra_date").isNotNull())
        .distinct()
    )

    df2_base = (
        df2
        .select(
            "msno",
            F.to_date(
                F.col("safra").cast("string"),
                "yyyyMM"
            ).alias("safra_date")
        )
        .filter(F.col("safra_date").isNotNull())
        .distinct()
    )

    df3_base = (
        df3
        .select(
            "msno",
            F.to_date(
                F.col("safra").cast("string"),
                "yyyyMM"
            ).alias("safra_date")
        )
        .filter(F.col("safra_date").isNotNull())
        .distinct()
    )

    # ========================================================
    # 3. CLIENTES PRESENTES NAS TRÊS BASES
    #
    # Aqui a presença também é considerada por SAFRA.
    # ========================================================

    candidatos = (
        df1_base
        .join(
            df2_base,
            on=["msno", "safra_date"],
            how="inner"
        )
        .join(
            df3_base,
            on=["msno", "safra_date"],
            how="inner"
        )
        .withColumn(
            "presenca",
            F.lit("111")
        )
    )

    # ========================================================
    # 4. CRIA AS SAFRAS HISTÓRICAS NECESSÁRIAS
    #
    # Para cada msno + safra:
    #
    # 201602
    #    ↓
    # 201511
    # 201512
    # 201601
    #
    # 201603
    #    ↓
    # 201512
    # 201601
    # 201602
    # ========================================================

    candidatos_historico = (
        candidatos
        .withColumn(
            "safras_historico",
            F.expr(
                f"""
                transform(
                    sequence(
                        1,
                        {meses_historico}
                    ),
                    x -> add_months(
                        safra_date,
                        -x
                    )
                )
                """
            )
        )
        .withColumn(
            "safra_historico",
            F.explode("safras_historico")
        )
        .drop("safras_historico")
    )

    # ========================================================
    # 5. VERIFICA HISTÓRICO EM TRANSACTIONS
    # ========================================================

    transactions_historico = (
        df2_base
        .withColumnRenamed("safra_date", "safra_historico") # Corrigido
        .withColumn(
            "tem_transactions",
            F.lit(1)
        )
    )

    hist_transactions = (
        candidatos_historico
        .join(
            transactions_historico,
            on=["msno", "safra_historico"],
            how="left"
        )
        .groupBy(
            "msno",
            "safra_date"
        )
        .agg(
            F.count(
                F.col("tem_transactions")
            ).alias(
                "qtd_meses_historico_transactions"
            )
        )
    )

    # ========================================================
    # 6. VERIFICA HISTÓRICO EM LOGS
    # ========================================================

    logs_historico = (
        df3_base
        .withColumnRenamed("safra_date", "safra_historico") # Corrigido
        .withColumn(
            "tem_logs",
            F.lit(1)
        )
    )

    hist_logs = (
        candidatos_historico
        .join(
            logs_historico,
            on=["msno", "safra_historico"],
            how="left"
        )
        .groupBy(
            "msno",
            "safra_date"
        )
        .agg(
            F.count(
                F.col("tem_logs")
            ).alias(
                "qtd_meses_historico_logs"
            )
        )
    )

    # ========================================================
    # 7. JUNTA AS INFORMAÇÕES DE HISTÓRICO
    # ========================================================

    merged = (
        candidatos

        .join(
            hist_transactions,
            on=["msno", "safra_date"],
            how="left"
        )

        .join(
            hist_logs,
            on=["msno", "safra_date"],
            how="left"
        )

        .fillna({
            "qtd_meses_historico_transactions": 0,
            "qtd_meses_historico_logs": 0
        })
    )

    # ========================================================
    # 8. CONVERTE SAFRA PARA YYYYMM
    # ========================================================

    merged = merged.withColumn(
        "safra",
        F.date_format(
            F.col("safra_date"),
            "yyyyMM"
        )
    )

    # ========================================================
    # 9. VALIDA HISTÓRICO COMPLETO
    # ========================================================

    merged = merged.withColumn(
        "historico_completo_transactions",
        F.when(
            F.col(
                "qtd_meses_historico_transactions"
            ) >= meses_historico,
            1
        ).otherwise(0)
    )

    merged = merged.withColumn(
        "historico_completo_logs",
        F.when(
            F.col(
                "qtd_meses_historico_logs"
            ) >= meses_historico,
            1
        ).otherwise(0)
    )

    # ========================================================
    # 10. DEFINE PÚBLICO-ALVO
    # ========================================================

    merged = merged.withColumn(
        "publico_alvo",
        F.when(
            (F.col("presenca") == "111") &
            (
                F.col(
                    "historico_completo_transactions"
                ) == 1
            ) &
            (
                F.col(
                    "historico_completo_logs"
                ) == 1
            ),
            1
        ).otherwise(0)
    )

    # ========================================================
    # 11. EXPLICA O MOTIVO DA ELEGIBILIDADE
    # ========================================================

    merged = merged.withColumn(
        "status_publico",

        F.when(
            F.col("historico_completo_transactions") == 0,
            "Sem histórico completo em transactions"
        )

        .when(
            F.col("historico_completo_logs") == 0,
            "Sem histórico completo em logs"
        )

        .otherwise(
            "Público-alvo"
        )
    )

    # ========================================================
    # 12. ORGANIZA COLUNAS
    # ========================================================

    colunas = [
        "msno",
        "safra",

        "presenca",

        "qtd_meses_historico_transactions",
        "historico_completo_transactions",

        "qtd_meses_historico_logs",
        "historico_completo_logs",

        "publico_alvo",
        "status_publico"
    ]

    merged = merged.select(colunas)

    # ========================================================
    # 13. RESUMO POR SAFRA
    # ========================================================

    print()
    print("=" * 70)
    print("📊 ELEGIBILIDADE POR SAFRA")
    print("=" * 70)

    resumo = (
        merged
        .groupBy("safra", "publico_alvo")
        .agg(
            F.countDistinct("msno").alias(
                "qtd_clientes"
            )
        )
        .orderBy(
            "safra",
            "publico_alvo"
        )
    )

    resumo.show(truncate=False)

    # ========================================================
    # 14. RESUMO DO PÚBLICO-ALVO
    # ========================================================

    print()
    print("=" * 70)
    print("🎯 RESUMO DO PÚBLICO-ALVO")
    print("=" * 70)

    resumo_publico = (
        merged
        .groupBy(
            "safra",
            "status_publico"
        )
        .agg(
            F.countDistinct("msno").alias(
                "qtd_clientes"
            )
        )
        .orderBy(
            "safra",
            "status_publico"
        )
    )

    resumo_publico.show(truncate=False)

    # ========================================================
    # 15. RETORNO
    # ========================================================

    return merged, resumo, resumo_publico

merged_ids, resumo, resumo_publico = analisar_bases(
    dfs_members,
    dfs_transactions,
    dfs_logs,
    nome1="members",
    nome2="transactions",
    nome3="logs",
    safra_modelagem_inicial="201601",
    meses_historico=3
)

📊 ANÁLISE DE INTERSECÇÃO DAS BASES
📅 Primeira safra de modelagem : 201601
📅 Histórico mínimo necessário : 3 meses
📅 Para 201601, histórico: ['201510', '201511', '201512']

📊 ELEGIBILIDADE POR SAFRA
+------+------------+------------+
|safra |publico_alvo|qtd_clientes|
+------+------------+------------+
|201601|0           |270385      |
|201601|1           |376725      |
|201602|0           |200744      |
|201602|1           |398075      |
|201603|0           |188848      |
|201603|1           |405053      |
|201604|0           |174595      |
|201604|1           |420866      |
|201605|0           |166403      |
|201605|1           |439530      |
|201606|0           |165061      |
|201606|1           |455270      |
|201607|0           |258105      |
|201607|1           |463250      |
|201608|0           |273655      |
|201608|1           |474992      |
|201609|0           |276251      |
|201609|1           |483174      |
|201610|0           |215800      |
|201610|1           |568488     

In [ ]:
publico_alvo = (
    merged_ids
    .filter(F.col("publico_alvo") == 1)
)

qtd_elegiveis = publico_alvo.select(IDENTIFY).distinct().count()

print(f"👥 Clientes elegíveis: {qtd_elegiveis:,}")

shutil.rmtree(output_analysis_path, ignore_errors=True)

(
    publico_alvo
    .write
    .mode("overwrite")
    .parquet(output_analysis_path)
)

print(f"💾 Público-alvo salvo em:")
print(f"   {output_analysis_path}")
print("✅ Apenas clientes elegíveis foram salvos!")

👥 Clientes elegíveis: 790,075
💾 Público-alvo salvo em:
   /content/drive/MyDrive/SANTANDER/merged_ids_analysis.parquet
✅ Apenas clientes elegíveis foram salvos!


In [ ]:
# Ativa a tolerância a falhas para arquivos corrompidos no Drive
spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")

ids_commons = spark.read.parquet(output_analysis_path)
ids_commons.show(10, truncate=False)

+--------------------------------------------+------+--------+--------------------------------+-------------------------------+------------------------+-----------------------+------------+--------------+
|msno                                        |safra |presenca|qtd_meses_historico_transactions|historico_completo_transactions|qtd_meses_historico_logs|historico_completo_logs|publico_alvo|status_publico|
+--------------------------------------------+------+--------+--------------------------------+-------------------------------+------------------------+-----------------------+------------+--------------+
|++CzVs35NSfU/gjDrGbWLVMweubEiMZsuvd3eTbXs58=|201607|111     |3                               |1                              |3                       |1                      |1           |Público-alvo  |
|++Ll4H/bT6Iiq/+0xcVwubXnuf0XR0MWVh2AnHAgxa8=|201612|111     |3                               |1                              |3                       |1                      |1   

# Membros

In [ ]:
def filtrar_historico_continuo(df: DataFrame) -> DataFrame:

    """
    Mantém somente clientes que possuem histórico contínuo de safras.

    Exemplo válido:
        201601
        201602
        201603
        201604

    Exemplo inválido:
        201601
        201602
        201604
        201605

    No segundo caso, 201603 está faltando.

    A regra é:
        quantidade de safras observadas
        ==
        quantidade de meses esperados entre a primeira
    """
    print("[PRE-PROCESSAMENTO] Verificando continuidade das safras...")

    # ------------------------------------------------------------
    # Identifica a primeira e a última safra de cada cliente.
    # ------------------------------------------------------------

    historico = (
        df
        .groupBy(IDENTIFY)
        .agg(
            F.min("safra_date_dt").alias("primeira_safra"),
            F.max("safra_date_dt").alias("ultima_safra"),
            F.countDistinct("safra_date_dt").alias("qtd_safras")
        )
    )

    # ------------------------------------------------------------
    # Calcula quantas safras deveriam existir entre a primeira
    # e a última safra.
    #
    # Exemplo:
    #
    # 201601 -> 201603
    #
    # Esperamos:
    # 201601
    # 201602
    # 201603
    #
    # Portanto: 3 safras.
    # ------------------------------------------------------------

    historico = historico.withColumn(
        "qtd_safras_esperadas",
        F.floor(
            F.months_between(
                F.col("ultima_safra"),
                F.col("primeira_safra")
            )
        ) + 1
    )

    # ------------------------------------------------------------
    # Cliente é considerado contínuo quando:
    #
    # quantidade de safras observadas
    # =
    # quantidade de safras esperadas.
    # ------------------------------------------------------------

    clientes_continuos = (
        historico
        .filter(
            F.col("qtd_safras") ==
            F.col("qtd_safras_esperadas")
        )
        .select(IDENTIFY)
    )

    # ------------------------------------------------------------
    # Estatísticas antes do filtro
    # ------------------------------------------------------------

    clientes_antes = (
        df
        .select(IDENTIFY)
        .distinct()
        .count()
    )

    # ------------------------------------------------------------
    # Mantém somente os clientes contínuos.
    # ------------------------------------------------------------

    df = df.join(
        clientes_continuos,
        on=IDENTIFY,
        how="inner"
    )

    # ------------------------------------------------------------
    # Estatísticas depois do filtro
    # ------------------------------------------------------------

    clientes_depois = (
        df
        .select(IDENTIFY)
        .distinct()
        .count()
    )

    clientes_removidos = clientes_antes - clientes_depois

    print(f"  Clientes antes       : {clientes_antes:,}")
    print(f"  Clientes depois      : {clientes_depois:,}")
    print(f"  Clientes removidos   : {clientes_removidos:,}")

    if clientes_antes > 0:
        percentual = (
            clientes_removidos / clientes_antes
        ) * 100

        print(f"  % removido            : {percentual:.2f}%")

    return df

### Consistência

In [ ]:
def preprocess_members(df):
    """
    ============================================================
    ETAPA 1 - PRÉ-PROCESSAMENTO
    ============================================================

    Pré-processa a tabela de members.

    Etapas:
    1. Mantém somente clientes presentes nas três bases;
    2. Corrige tipos das colunas;
    3. Trata valores ausentes de gender;
    4. Cria as colunas de data;
    5. Remove registros cuja data de cadastro é posterior à safra;
    6. Mantém somente o histórico contínuo do cliente.

    Observação sobre bd:
    ------------------------------------------------------------
    A variável 'bd' representa idade.

    A base original possui valores potencialmente problemáticos,
    incluindo valores negativos e valores muito elevados.

    Neste pipeline NÃO é realizado nenhum tratamento estatístico
    ou remoção de outliers em 'bd'.

    Essa análise será realizada posteriormente, após uma análise
    estatística específica da variável.
    """

    print("=" * 60)
    print("ETAPA 1 - PRÉ-PROCESSAMENTO - MEMBERS")
    print("=" * 60)

    ############################################################
    # 1. Registros iniciais
    ############################################################

    print(
        f"[PRE-PROCESSAMENTO] Registros iniciais: "
        f"{df.count():,}"
    )

    ############################################################
    # 2. Mantém somente clientes presentes nas três bases
    #
    # O arquivo merged_ids_analysis.parquet foi criado
    # previamente através da análise de interseção das bases.
    #
    # Portanto, utilizamos esse universo para evitar
    # processar clientes que posteriormente seriam eliminados
    # na construção da MASTER.
    ############################################################

    ids_validos = (
        spark.read.parquet(
            output_analysis_path
        )
        .select("msno")
        .distinct()
    )

    ############################################################
    # 3. Filtro por clientes válidos
    ############################################################

    df = df.join(
        ids_validos,
        on="msno",
        how="inner"
    )

    print(
        f"[PRE-PROCESSAMENTO] Registros após filtro "
        f"de clientes presentes nas três bases: "
        f"{df.count():,}"
    )

    ############################################################
    # 4. VALORES AUSENTES
    ############################################################

    # Gender:
    # ausência de informação é mantida explicitamente como
    # uma categoria "unknown".
    #
    # Não utilizamos moda ou qualquer outra imputação,
    # pois a ausência pode possuir informação própria.

    if "gender" in df.columns:

        df = df.withColumn(
            "gender",
            F.coalesce(
                F.col("gender"),
                F.lit("unknown")
            )
        )

    ############################################################
    # 5. CONVERSÃO DE TIPOS
    ############################################################

    colunas_int = [
        "city",
        "bd",
        "registered_via",
        "is_ativo"
    ]

    for coluna in colunas_int:

        if coluna in df.columns:

            df = df.withColumn(
                coluna,
                F.col(coluna).cast("int")
            )

    ############################################################
    # 6. REGISTRATION INIT TIME
    ############################################################

    if "registration_init_time" not in df.columns:
        raise ValueError(
            "Coluna 'registration_init_time' não encontrada."
        )

    df = df.withColumn(
        "registration_init_time",
        F.to_date(
            F.col("registration_init_time").cast("string"),
            "yyyyMMdd"
        )
    )

    ############################################################
    # 7. SAFRA
    ############################################################

    if "safra" not in df.columns:
        raise ValueError(
            "Coluna 'safra' não encontrada."
        )

    df = df.withColumn(
        "safra",
        F.col("safra").cast("string")
    )

    df = df.withColumn(
        "safra_date_dt",
        F.to_date(
            F.col("safra"),
            "yyyyMM"
        )
    )

    ############################################################
    # 8. REMOVE INCONSISTÊNCIAS
    #
    # Não faz sentido um cliente possuir uma data de cadastro
    # posterior à safra na qual ele está sendo observado.
    #
    # Exemplo inválido:
    #
    # registration_init_time = 2017-01-01
    # safra                  = 2016-09
    #
    # Portanto, removemos essas observações.
    ############################################################

    df = df.filter(
        F.col("registration_init_time")
        <=
        F.col("safra_date_dt")
    )

    print(
        "[PRE-PROCESSAMENTO] Registros após filtro "
        "de inconsistências temporais: "
        f"{df.count():,}"
    )

    ############################################################
    # 9. HISTÓRICO CONTÍNUO
    ############################################################

    df = filtrar_historico_continuo(df)

    print(
        "[PRE-PROCESSAMENTO] Registros após filtro "
        "de safras sequentes: "
        f"{df.count():,}"
    )

    return df

### Features

In [ ]:
def build_temporal_features(df):
    """
    ==============================================================
    ETAPA 2 - FEATURES TEMPORAIS
    ==============================================================

    Nesta etapa são criadas apenas variáveis relacionadas ao tempo
    de relacionamento do cliente.

    Features criadas
    ----------------
    primeira_safra_msno
        Primeira vez que o cliente apareceu na base histórica.

    tempo_cliente_meses
        Quantidade de meses entre a data de cadastro
        e a safra atual.

    Nenhuma feature de comportamento é criada aqui.
    """

    print("=" * 60)
    print("ETAPA 2 - FEATURES TEMPORAIS")
    print("=" * 60)

    ###############################################################
    # Janela contendo todas as linhas do cliente
    ###############################################################

    w_cliente = Window.partitionBy(IDENTIFY)

    ###############################################################
    # Primeira safra observada do cliente
    #
    # Não necessariamente é a data de cadastro.
    #
    # É apenas a primeira vez que o cliente aparece
    # na base histórica.
    ###############################################################

    df = df.withColumn(
        "primeira_safra_msno",
        F.min("safra_date_dt").over(w_cliente)
    )

    ###############################################################
    # Tempo de relacionamento
    #
    # Mede quantos meses se passaram desde o cadastro.
    #
    # Exemplo:
    #
    # cadastro: jan/2016
    # safra:    mar/2016
    #
    # resultado = 3 meses
    #
    # (+1 porque o mês do cadastro conta como
    # o primeiro mês de relacionamento)
    ###############################################################

    df = df.withColumn(
        "tempo_cliente_meses",
        (
            F.floor(
                F.months_between(
                    F.col("safra_date_dt"),
                    F.col("registration_init_time")
                )
            ) + 1
        ).cast("int")
    )

    ###############################################################
    # Garantia de consistência
    #
    # Não existem clientes com tempo <= 0.
    #
    # Caso a data de cadastro seja NULL, o tempo de relacionamento
    # também permanecerá NULL.
    ###############################################################

    df = df.withColumn(
        "tempo_cliente_meses",
        F.when(
            F.col("tempo_cliente_meses").isNull(),
            F.lit(None).cast("int")
        )
        .when(
            F.col("tempo_cliente_meses") < 1,
            1
        )
        .otherwise(
            F.col("tempo_cliente_meses")
        )
    )

    ###############################################################
    # Estatísticas rápidas
    ###############################################################

    print("Tempo de relacionamento:")

    (
        df
        .agg(
            F.avg(
                "tempo_cliente_meses"
            ).alias("media"),

            F.min(
                "tempo_cliente_meses"
            ).alias("minimo"),

            F.max(
                "tempo_cliente_meses"
            ).alias("maximo")
        )
        .show()
    )

    print("Features criadas:")

    print(" - primeira_safra_msno")
    print(" - tempo_cliente_meses")

    return df

### Target

In [ ]:
def filtrar_horizonte_completo(df):
    """
    Mantém somente observações que possuem os três meses
    futuros necessários para calcular o churn_m3.

    Observações sem horizonte completo não são consideradas
    no conjunto de modelagem.
    """

    return df.filter(
        (F.col("_horizonte_completo") == 1) &
        F.col("churn_m3").isNotNull()
    )

def build_target_churn(df):
    """
    ============================================================
    ETAPA 3 - CONSTRUÇÃO DO TARGET DE CHURN
    ============================================================

    Definição:
    ------------------------------------------------------------
    Uma observação é elegível quando:

        1. O cliente está ativo na safra atual (t);
        2. Existe pelo menos uma safra futura consecutiva.

    O horizonte de observação considera até os três meses
    seguintes:

        t+1
        t+2
        t+3

    Não é necessário que existam os três meses.

    Exemplos:

        t+1 existe              → pode calcular
        t+1 e t+2 existem       → pode calcular
        t+1, t+2 e t+3 existem  → pode calcular

    churn_m3 = 1 quando o cliente fica inativo em pelo menos
    uma das safras futuras observadas.

    churn_m3 = 0 quando permanece ativo em todas as safras
    futuras observadas.

    Observações:
        - cliente já inativo em t;
        - sem nenhuma safra futura;

    não são elegíveis para modelagem e posteriormente são
    removidas.
    """

    ############################################################
    # 1. Janela temporal
    ############################################################

    w = (
        Window
        .partitionBy(IDENTIFY)
        .orderBy("safra_date_dt")
    )

    ############################################################
    # 2. Recupera as três safras futuras
    ############################################################

    df = (
        df
        .withColumn(
            "_safra_t1",
            F.lead("safra_date_dt", 1).over(w)
        )
        .withColumn(
            "_safra_t2",
            F.lead("safra_date_dt", 2).over(w)
        )
        .withColumn(
            "_safra_t3",
            F.lead("safra_date_dt", 3).over(w)
        )
        .withColumn(
            "_ativo_t1",
            F.lead("is_ativo", 1).over(w)
        )
        .withColumn(
            "_ativo_t2",
            F.lead("is_ativo", 2).over(w)
        )
        .withColumn(
            "_ativo_t3",
            F.lead("is_ativo", 3).over(w)
        )
    )

    ############################################################
    # 3. Verifica quais meses futuros são realmente consecutivos
    ############################################################

    t1_valido = (
        F.col("_safra_t1") ==
        F.add_months(F.col("safra_date_dt"), 1)
    )

    t2_valido = (
        F.col("_safra_t2") ==
        F.add_months(F.col("safra_date_dt"), 2)
    )

    t3_valido = (
        F.col("_safra_t3") ==
        F.add_months(F.col("safra_date_dt"), 3)
    )

    ############################################################
    # 4. Pelo menos um mês futuro disponível
    ############################################################

    horizonte_completo = (
        t1_valido |
        t2_valido |
        t3_valido
    )

    df = df.withColumn(
        "_horizonte_completo",
        horizonte_completo.cast("int")
    )

    ############################################################
    # 5. Verifica se houve inatividade em algum mês futuro
    #
    # Só consideramos os meses que realmente existem.
    ############################################################

    houve_inatividade = (
        (t1_valido & (F.col("_ativo_t1") == 0))
        |
        (t2_valido & (F.col("_ativo_t2") == 0))
        |
        (t3_valido & (F.col("_ativo_t3") == 0))
    )

    ############################################################
    # 6. Construção do target
    ############################################################

    df = df.withColumn(
        "churn_m3",

        F.when(
            (F.col("is_ativo") == 1) &
            horizonte_completo &
            houve_inatividade,
            F.lit(1)
        )

        .when(
            (F.col("is_ativo") == 1) &
            horizonte_completo,
            F.lit(0)
        )

        .otherwise(
            F.lit(None).cast("integer")
        )
    )

    ############################################################
    # 7. Remove colunas auxiliares
    ############################################################

    df = df.drop(
        "_safra_t1",
        "_safra_t2",
        "_safra_t3",
        "_ativo_t1",
        "_ativo_t2",
        "_ativo_t3"
    )

    return df

### Behavior

In [ ]:
def build_behavior_features(df):
    """
    ============================================================
    ETAPA 4 - FEATURES DE COMPORTAMENTO
    ============================================================

    Objetivo
    --------
    Construir variáveis que descrevem o comportamento recente
    do cliente utilizando APENAS informações disponíveis até a
    safra atual.

    Nenhuma feature utiliza informações do futuro.

    Apenas os targets utilizam informações futuras.
    """

    print("=" * 60)
    print("ETAPA 4 - FEATURES DE COMPORTAMENTO")
    print("=" * 60)

    #################################################################
    # Janela temporal do cliente
    #
    # Todas as features são calculadas respeitando a ordem
    # cronológica das safras de cada cliente.
    #################################################################

    w = (
        Window
        .partitionBy(IDENTIFY)
        .orderBy("safra_date_dt")
    )

    #################################################################
    # Janela contendo registros anteriores E a safra atual.
    #
    # Utilizada para descobrir qual foi a última safra em que
    # o cliente esteve ativo até o momento atual.
    #
    # Dessa forma:
    #
    # Cliente ativo na safra atual:
    #     ultima_safra_ativa = safra atual
    #
    # Cliente inativo na safra atual:
    #     ultima_safra_ativa = última safra ativa anterior
    #################################################################

    janela_ate_atual = (
        Window
        .partitionBy(IDENTIFY)
        .orderBy("safra_date_dt")
        .rowsBetween(
            Window.unboundedPreceding,
            Window.currentRow
        )
    )

    #################################################################
    # 1. Última safra ativa
    #
    # Exemplo:
    #
    # Jan  ativo
    # Fev  ativo
    # Mar  inativo
    #
    # Resultado:
    #
    # Jan → Jan
    # Fev → Fev
    # Mar → Fev
    #
    # A safra atual é considerada.
    #################################################################

    df = df.withColumn(
        "ultima_safra_ativa",
        F.max(
            F.when(
                F.col("is_ativo") == 1,
                F.col("safra_date_dt")
            )
        ).over(janela_ate_atual)
    )

    #################################################################
    # 2. Meses desde a última atividade
    #
    # Mede há quantos meses o cliente não apresenta atividade.
    #
    # Cliente ativo:
    #     0
    #
    # Cliente inativo:
    #     diferença entre a safra atual e a última safra ativa.
    #
    # Cliente que nunca esteve ativo:
    #     utiliza a primeira safra observada.
    #################################################################

    df = df.withColumn(
        "meses_desde_ultima_atividade",
        F.when(
            F.col("ultima_safra_ativa").isNull(),
            F.floor(
                F.months_between(
                    F.col("safra_date_dt"),
                    F.col("primeira_safra_msno")
                )
            )
        )
        .otherwise(
            F.floor(
                F.months_between(
                    F.col("safra_date_dt"),
                    F.col("ultima_safra_ativa")
                )
            )
        )
    )

    #################################################################
    # 3. Cliente reativado
    #
    # Marca quando o cliente estava inativo na safra anterior
    # e voltou a ficar ativo na safra atual.
    #
    # Primeira safra:
    #     não possui safra anterior → 0
    #################################################################

    ativo_mes_anterior = F.lag(
        "is_ativo"
    ).over(w)

    df = df.withColumn(
        "cliente_reativado",
        F.when(
            (ativo_mes_anterior == 0)
            &
            (F.col("is_ativo") == 1),
            1
        ).otherwise(0)
    )

    #################################################################
    # 4. Cliente esteve ativo nos últimos 2 meses?
    #
    # A janela inclui a safra atual.
    #
    # Exemplo:
    #
    # Jan 0
    # Fev 1
    #
    # Resultado em fevereiro = 1
    #
    # Jan 0
    # Fev 0
    #
    # Resultado = 0
    #################################################################

    df = df.withColumn(
        "ativo_ultimos_2_meses",
        F.max("is_ativo").over(
            w.rowsBetween(-1, 0)
        )
    )

    #################################################################
    # 5. Cliente esteve ativo nos últimos 3 meses?
    #
    # A janela inclui a safra atual.
    #################################################################

    df = df.withColumn(
        "ativo_ultimos_3_meses",
        F.max("is_ativo").over(
            w.rowsBetween(-2, 0)
        )
    )

    #################################################################
    # 6. Quantidade de meses ativos
    #
    # Diferente das flags acima.
    #
    # Conta quantas das últimas 3 safras, incluindo a atual,
    # possuem is_ativo = 1.
    #
    # Exemplo:
    #
    # Histórico:
    #
    # 1 1 0
    #
    # Resultado = 2
    #################################################################

    df = df.withColumn(
        "qtd_meses_ativos_ultimos_3_meses",
        F.sum("is_ativo").over(
            w.rowsBetween(-2, 0)
        )
    )

    #################################################################
    # 7. Meses consecutivos inativos
    #
    # Exemplo:
    #
    # Ativo
    #
    # 1 1 0 0 0 1 0 0
    #
    # Resultado:
    #
    # 0 0 1 2 3 0 1 2
    #
    # A estratégia consiste em:
    #
    # 1) identificar quando o status muda;
    # 2) numerar cada bloco contínuo;
    # 3) contar apenas os registros inativos.
    #################################################################

    status_anterior = F.lag(
        "is_ativo"
    ).over(w)

    df = df.withColumn(
        "_mudou_status",
        F.when(
            status_anterior.isNull(),
            0
        )
        .when(
            F.col("is_ativo") != status_anterior,
            1
        )
        .otherwise(0)
    )

    #################################################################
    # Cria um identificador acumulado para cada bloco de status.
    #################################################################

    w_acumulada = (
        Window
        .partitionBy(IDENTIFY)
        .orderBy("safra_date_dt")
        .rowsBetween(
            Window.unboundedPreceding,
            Window.currentRow
        )
    )

    df = df.withColumn(
        "_bloco_status",
        F.sum(
            "_mudou_status"
        ).over(w_acumulada)
    )

    #################################################################
    # Janela de cada bloco contínuo
    #################################################################

    janela_bloco = (
        Window
        .partitionBy(
            IDENTIFY,
            "_bloco_status"
        )
        .orderBy("safra_date_dt")
    )

    df = df.withColumn(
        "meses_consecutivos_inativos",
        F.when(
            F.col("is_ativo") == 0,
            F.row_number().over(janela_bloco)
        )
        .otherwise(0)
    )

    #################################################################
    # Remove colunas auxiliares
    #################################################################

    df = df.drop(
        "_mudou_status",
        "_bloco_status"
    )

    #################################################################
    # Resumo das features criadas
    #################################################################

    print("Features criadas:")

    features = [
        "ultima_safra_ativa",
        "meses_desde_ultima_atividade",
        "cliente_reativado",
        "ativo_ultimos_2_meses",
        "ativo_ultimos_3_meses",
        "qtd_meses_ativos_ultimos_3_meses",
        "meses_consecutivos_inativos"
    ]

    for feature in features:
        print(f"  ✓ {feature}")

    return df

### Define Público Alvo

In [ ]:
def filtrar_clientes_historico_minimo(df, minimo_safras):
    """
    Mantém apenas clientes que possuem pelo menos
    'minimo_safras' safras distintas no histórico.

    A contagem é feita sobre safra_date_dt para garantir
    que estamos contando períodos temporais distintos,
    e não simplesmente quantidade de linhas.
    """

    clientes_validos = (
        df
        .groupBy(IDENTIFY)
        .agg(
            F.countDistinct(
                "safra_date_dt"
            ).alias("qtd_safras")
        )
        .filter(
            F.col("qtd_safras") >= minimo_safras
        )
        .select(IDENTIFY)
    )

    return df.join(
        clientes_validos,
        on=IDENTIFY,
        how="inner"
    )

def filtrar_clientes_que_comecam_ativos(df):
    """
    Mantém apenas clientes cuja primeira safra observada
    já possui is_ativo = 1.

    Isso garante que o modelo trabalhe com clientes que
    iniciam sua trajetória histórica como clientes ativos.
    """

    w = (
        Window
        .partitionBy(IDENTIFY)
        .orderBy("safra_date_dt")
    )

    clientes_validos = (
        df
        .withColumn(
            "_rn",
            F.row_number().over(w)
        )
        .filter(
            (F.col("_rn") == 1) &
            (F.col("is_ativo") == 1)
        )
        .select(IDENTIFY)
        .distinct()
    )

    return df.join(
        clientes_validos,
        on=IDENTIFY,
        how="inner"
    )

def filtrar_ate_primeiro_churn(df):
    """
    Mantém somente o histórico do cliente até o
    primeiro churn observado.

    A própria safra em que churn_m3 = 1 é mantida,
    pois ela representa uma observação válida para
    modelagem do churn.

    Exemplo:

        201601 -> ativo
        201602 -> ativo
        201603 -> churn_m3 = 1
        201604 -> removido
        201605 -> removido

    O cliente permanece na base até 201603.
    """

    w_cliente = Window.partitionBy(IDENTIFY)

    df = df.withColumn(
        "_primeira_safra_churn",
        F.min(
            F.when(
                F.col("churn_m3") == 1,
                F.col("safra_date_dt")
            )
        ).over(w_cliente)
    )

    df = df.filter(
        F.col("_primeira_safra_churn").isNull()
        |
        (
            F.col("safra_date_dt")
            <= F.col("_primeira_safra_churn")
        )
    )

    return df.drop(
        "_primeira_safra_churn"
    )
def get_publico_alvo(
    df,
    safra_limite="201609",
    minimo_safras=4
):
    """
    Constrói o público-alvo para modelagem de churn.

    Regras:
    1. Remove clientes que começam inativos;
    2. Mantém apenas as safras até o período de modelagem;
    3. Exige um histórico mínimo de safras;
    4. Mantém o histórico somente até o primeiro churn;
    5. Mantém somente observações com horizonte futuro completo.

    IMPORTANTE:
    ------------------------------------------------------------
    O filtro de histórico mínimo é realizado por CLIENTE.
    Portanto, quando um cliente é aprovado, todas as suas
    observações são mantidas.

    O filtro de horizonte ocorre somente no final, depois que
    as features temporais e comportamentais já foram calculadas.
    """

    print("=" * 70)
    print("Construindo público-alvo")
    print("=" * 70)

    print(f"Linhas iniciais : {df.count():,}")
    print(
        f"Clientes iniciais: "
        f"{df.select(IDENTIFY).distinct().count():,}"
    )

    ############################################################
    # 1. Clientes que começam ativos
    ############################################################

    df = filtrar_clientes_que_comecam_ativos(df)

    print("\n[1] Clientes que começam ativos")
    print(f"Linhas   : {df.count():,}")
    print(
        f"Clientes : "
        f"{df.select(IDENTIFY).distinct().count():,}"
    )

    ############################################################
    # 2. Limite temporal da modelagem
    ############################################################

    df = df.filter(
        F.col("safra") <= safra_limite
    )

    print(f"\n[2] Safras <= {safra_limite}")
    print(f"Linhas   : {df.count():,}")
    print(
        f"Clientes : "
        f"{df.select(IDENTIFY).distinct().count():,}"
    )

    ############################################################
    # 3. Histórico mínimo
    ############################################################

    df = filtrar_clientes_historico_minimo(
        df,
        minimo_safras=minimo_safras
    )

    print(
        f"\n[3] Clientes com pelo menos "
        f"{minimo_safras} safras"
    )

    print(f"Linhas   : {df.count():,}")
    print(
        f"Clientes : "
        f"{df.select(IDENTIFY).distinct().count():,}"
    )

    ############################################################
    # 4. Histórico até o primeiro churn
    ############################################################

    df = filtrar_ate_primeiro_churn(df)

    print("\n[4] Histórico até o primeiro churn")
    print(f"Linhas   : {df.count():,}")
    print(
        f"Clientes : "
        f"{df.select(IDENTIFY).distinct().count():,}"
    )

    ############################################################
    # 5. Somente observações com horizonte completo
    ############################################################

    df = filtrar_horizonte_completo(df)

    print("\n[5] Observações com horizonte completo")
    print(f"Linhas   : {df.count():,}")
    print(
        f"Clientes : "
        f"{df.select(IDENTIFY).distinct().count():,}"
    )

    ############################################################
    # 6. Remove coluna auxiliar
    ############################################################

    df = df.drop("_horizonte_completo")

    print("\n✅ Público-alvo criado com sucesso!")

    return df

### Pipeline Main - MEMBERS

In [ ]:
%%time
def run_pipeline(
    input_path: str,
    output_path: str,
    overwrite: bool = True
) -> DataFrame:

    """
    ============================================================
    PIPELINE DE FEATURE ENGINEERING - MEMBERS
    ============================================================

    Fluxo:

    1. Leitura
    2. Pré-processamento
    3. Features temporais
    4. Features de comportamento
    5. Construção do target
    6. Construção do público-alvo
    7. Ordenação
    8. Salvamento
    9. Retorno

    IMPORTANTE
    ------------------------------------------------------------
    O target churn_m3 utiliza informações dos três meses
    posteriores à safra atual.

    Portanto, o dataset NÃO deve ser limitado a 201609 antes
    da construção do target.

    Para uma observação de 201609, por exemplo, precisamos
    observar:

        201610
        201611
        201612

    para determinar corretamente o churn.

    O limite 201609 é aplicado somente na construção do
    público-alvo final.
    """

    print("=" * 70)
    print("PIPELINE DE FEATURE ENGINEERING - MEMBERS")
    print("=" * 70)

    # ==========================================================
    # 1. LEITURA
    # ==========================================================

    print("\n[1/8] Lendo dataset...")

    df = spark.read.parquet(input_path)

    print(f"Registros: {df.count():,}")
    print(f"Colunas: {len(df.columns)}")

    # ==========================================================
    # 2. PRÉ-PROCESSAMENTO
    # ==========================================================

    print("\n[2/8] Pré-processamento...")

    df = preprocess_members(df)

    print(
        f"Registros após preprocessamento: "
        f"{df.count():,}"
    )

    # ==========================================================
    # 3. FEATURES TEMPORAIS
    # ==========================================================

    print("\n[3/8] Construindo features temporais...")

    df = build_temporal_features(df)

    # ==========================================================
    # 4. FEATURES DE COMPORTAMENTO
    # ==========================================================

    print("\n[4/8] Construindo features de comportamento...")

    df = build_behavior_features(df)

    # ==========================================================
    # 5. TARGET
    # ==========================================================

    print("\n[5/8] Construindo target de churn...")

    df = build_target_churn(df)

    # ==========================================================
    # DIAGNÓSTICO DO TARGET
    # ==========================================================

    print("\n========== DIAGNÓSTICO DO TARGET ==========")

    (
        df
        .groupBy("churn_m3")
        .count()
        .orderBy("churn_m3")
        .show()
    )

    # ==========================================================
    # 6. PÚBLICO-ALVO
    # ==========================================================

    print("\n[6/8] Construindo público-alvo...")

    df = get_publico_alvo(
        df,
        safra_limite="201609",
        minimo_safras=4
    )

    # ==========================================================
    # 7. ORDENAÇÃO FINAL
    # ==========================================================

    print("\n[7/8] Ordenando dataset...")

    df = df.orderBy(
        IDENTIFY,
        "safra_date_dt"
    )

    # ==========================================================
    # 8. SALVAMENTO
    # ==========================================================

    print("\n[8/8] Salvando dataset...")

    if overwrite:
        shutil.rmtree(
            output_path,
            ignore_errors=True
        )

    (
        df
        .write
        .mode(
            "overwrite" if overwrite else "error"
        )
        .parquet(output_path)
    )

    # ==========================================================
    # RESUMO FINAL
    # ==========================================================

    print("\n" + "=" * 70)
    print("PIPELINE CONCLUÍDO COM SUCESSO")
    print("=" * 70)

    print(
        f"Registros finais : "
        f"{df.count():,}"
    )

    print(
        f"Clientes finais  : "
        f"{df.select(IDENTIFY).distinct().count():,}"
    )

    print(
        f"Colunas finais   : "
        f"{len(df.columns)}"
    )

    print("\nDistribuição do target:")

    (
        df
        .groupBy("churn_m3")
        .count()
        .orderBy("churn_m3")
        .show()
    )

    print("\nPeríodo final:")

    (
        df
        .select(
            F.min("safra").alias("safra_min"),
            F.max("safra").alias("safra_max")
        )
        .show()
    )

    print("\nDataset salvo em:")
    print(output_path)

    return df

input_path = "/content/drive/MyDrive/SANTANDER/members.parquet"
output_path = "/content/drive/MyDrive/SANTANDER/features_members.parquet"

dfs_members = run_pipeline(input_path, output_path, True)

PIPELINE DE FEATURE ENGINEERING - MEMBERS

[1/8] Lendo dataset...
Registros: 63,867,246
Colunas: 8

[2/8] Pré-processamento...
ETAPA 1 - PRÉ-PROCESSAMENTO - MEMBERS
[PRE-PROCESSAMENTO] Registros iniciais: 63,867,246
[PRE-PROCESSAMENTO] Registros após filtro de clientes presentes nas três bases: 9,031,184
[PRE-PROCESSAMENTO] Registros após filtro de inconsistências temporais: 8,900,836
[PRE-PROCESSAMENTO] Verificando continuidade das safras...
  Clientes antes       : 790,075
  Clientes depois      : 790,075
  Clientes removidos   : 0
  % removido            : 0.00%
[PRE-PROCESSAMENTO] Registros após filtro de safras sequentes: 8,900,836
Registros após preprocessamento: 8,900,836

[3/8] Construindo features temporais...
ETAPA 2 - FEATURES TEMPORAIS
Tempo de relacionamento:
+-----------------+------+------+
|            media|minimo|maximo|
+-----------------+------+------+
|41.50451182338378|     1|   153|
+-----------------+------+------+

Features criadas:
 - primeira_safra_msno
 - te

# Logs

### Consistência

In [ ]:
def preprocess_logs(df: DataFrame) -> DataFrame:
    """
    ============================================================
    ETAPA 1 - PRÉ-PROCESSAMENTO - LOGS
    ============================================================

    Responsabilidades:

    1. Validar as colunas obrigatórias;
    2. Padronizar identificador e safra;
    3. Criar a data da safra;
    4. Filtrar clientes presentes nas três bases;
    5. Converter variáveis numéricas;
    6. Tratar valores negativos;
    7. Tratar NULLs das métricas de consumo;
    8. Remover registros sem chave temporal.

    O histórico contínuo é tratado posteriormente pela função
    filtrar_historico_continuo().
    """

    print("=" * 60)
    print("ETAPA 1 - PRÉ-PROCESSAMENTO - LOGS")
    print("=" * 60)

    # ==========================================================
    # 0. REGISTROS INICIAIS
    # ==========================================================

    registros_iniciais = df.count()

    print(
        f"[PRE-PROCESSAMENTO] Registros iniciais: "
        f"{registros_iniciais:,}"
    )

    # ==========================================================
    # 1. VALIDAÇÃO DAS COLUNAS
    # ==========================================================

    colunas_obrigatorias = [
        IDENTIFY,
        TIME_COL,
        "num_25",
        "num_50",
        "num_75",
        "num_985",
        "num_100",
        "num_unq",
        "total_secs"
    ]

    colunas_faltantes = [
        c
        for c in colunas_obrigatorias
        if c not in df.columns
    ]

    if colunas_faltantes:

        raise ValueError(
            "Colunas obrigatórias ausentes no LOGS: "
            f"{colunas_faltantes}"
        )

    # ==========================================================
    # 2. IDENTIFICADOR
    # ==========================================================

    df = df.withColumn(
        IDENTIFY,
        F.trim(
            F.col(IDENTIFY).cast("string")
        )
    )

    # ==========================================================
    # 3. SAFRA
    # ==========================================================

    df = df.withColumn(
        TIME_COL,
        F.trim(
            F.col(TIME_COL).cast("string")
        )
    )

    # ==========================================================
    # 4. DATA DA SAFRA
    # ==========================================================

    df = df.withColumn(
        "safra_date_dt",
        F.to_date(
            F.col(TIME_COL),
            "yyyyMM"
        )
    )

    # ==========================================================
    # 5. VARIÁVEIS NUMÉRICAS
    # ==========================================================

    numeric_cols = [
        "num_25",
        "num_50",
        "num_75",
        "num_985",
        "num_100",
        "num_unq",
        "total_secs"
    ]

    for col_name in numeric_cols:

        # ------------------------------------------------------
        # Conversão para double
        # ------------------------------------------------------

        df = df.withColumn(
            col_name,
            F.col(col_name).cast("double")
        )

        # ------------------------------------------------------
        # Valores negativos
        # ------------------------------------------------------

        df = df.withColumn(
            col_name,
            F.when(
                F.col(col_name) < 0,
                F.lit(0.0)
            ).otherwise(
                F.col(col_name)
            )
        )

        # ------------------------------------------------------
        # NULL
        # ------------------------------------------------------

        df = df.withColumn(
            col_name,
            F.coalesce(
                F.col(col_name),
                F.lit(0.0)
            )
        )

    # ==========================================================
    # 6. REMOVER REGISTROS SEM CHAVE
    # ==========================================================

    df = df.dropna(
        subset=[
            IDENTIFY,
            TIME_COL,
            "safra_date_dt"
        ]
    )

    registros_apos_chave = df.count()

    print(
        f"[PRE-PROCESSAMENTO] Registros após "
        f"validação temporal: "
        f"{registros_apos_chave:,}"
    )

    # ==========================================================
    # 7. CLIENTES PRESENTES NAS TRÊS BASES
    # ==========================================================

    ids_validos = (
        spark.read.parquet(
            output_analysis_path
        )
        .select(
            IDENTIFY
        )
        .distinct()
    )

    df = df.join(
        ids_validos,
        on=IDENTIFY,
        how="inner"
    )

    registros_apos_filtro_clientes = df.count()

    print(
        f"[PRE-PROCESSAMENTO] Registros após filtro "
        f"de clientes presentes nas três bases: "
        f"{registros_apos_filtro_clientes:,}"
    )

    # ==========================================================
    # 8. RESUMO
    # ==========================================================

    print(
        f"[PRE-PROCESSAMENTO] Registros finais da etapa: "
        f"{registros_apos_filtro_clientes:,}"
    )

    return df

### Features

In [ ]:
# ==============================================================
# 2. AGREGAÇÃO MENSAL
# ==============================================================

def aggregate_logs_monthly(
    df: DataFrame
) -> DataFrame:

    """
    ============================================================
    AGREGAÇÃO MENSAL - LOGS
    ============================================================

    Unidade:

        1 cliente × 1 safra
    """

    print("  → Agregando logs por cliente e safra...")

    df_month = (
        df
        .groupBy(
            IDENTIFY,
            TIME_COL,
            "safra_date_dt"
        )
        .agg(

            # ==================================================
            # Volume de consumo
            # ==================================================

            F.sum("num_25").alias("num_25"),
            F.sum("num_50").alias("num_50"),
            F.sum("num_75").alias("num_75"),
            F.sum("num_985").alias("num_985"),
            F.sum("num_100").alias("num_100"),

            # ==================================================
            # Diversidade
            # ==================================================

            F.sum("num_unq").alias("num_unq"),

            # ==================================================
            # Tempo de consumo
            # ==================================================

            F.sum("total_secs").alias("total_secs"),

            # ==================================================
            # FREQUÊNCIA
            #
            # Conta quantos dias tiveram algum consumo.
            # ==================================================

            F.sum(
                F.when(
                    F.col("total_secs") > 0,
                    1
                ).otherwise(0)
            ).alias("dias_ativos")
        )
    )

    # ----------------------------------------------------------
    # Garantia adicional contra NULL
    # ----------------------------------------------------------

    metricas = [
        "num_25",
        "num_50",
        "num_75",
        "num_985",
        "num_100",
        "num_unq",
        "total_secs",
        "dias_ativos"
    ]

    for col_name in metricas:

        df_month = df_month.withColumn(
            col_name,
            F.coalesce(
                F.col(col_name),
                F.lit(0.0)
            )
        )

    # ==========================================================
    # TOTAL DE PLAYS
    # ==========================================================

    df_month = df_month.withColumn(
        "total_plays",
        (
            F.col("num_25") +
            F.col("num_50") +
            F.col("num_75") +
            F.col("num_985") +
            F.col("num_100")
        )
    )

    # ==========================================================
    # TAXA DE COMPLETUDE
    # ==========================================================

    df_month = df_month.withColumn(
        "taxa_completude",
        F.when(
            F.col("total_plays") > 0,
            F.col("num_100") /
            F.col("total_plays")
        ).otherwise(0.0)
    )

    # ==========================================================
    # SKIP RATIO
    # ==========================================================

    df_month = df_month.withColumn(
        "skip_ratio",
        F.when(
            F.col("total_plays") > 0,
            (
                F.col("num_25") +
                F.col("num_50")
            ) / F.col("total_plays")
        ).otherwise(0.0)
    )

    # ==========================================================
    # TEMPO MÉDIO POR PLAY
    # ==========================================================

    df_month = df_month.withColumn(
        "avg_secs_per_play",
        F.when(
            F.col("total_plays") > 0,
            F.col("total_secs") /
            F.col("total_plays")
        ).otherwise(0.0)
    )

    # ==========================================================
    # PROPORÇÃO DE MÚSICAS ÚNICAS
    # ==========================================================

    df_month = df_month.withColumn(
        "diversidade_ratio",
        F.when(
            F.col("total_plays") > 0,
            F.col("num_unq") /
            F.col("total_plays")
        ).otherwise(0.0)
    )

    return df_month

### Behavior

In [ ]:
# ==============================================================
# 3. FEATURES DE COMPORTAMENTO
# ==============================================================

def build_behavior_features_logs(
    df: DataFrame
) -> DataFrame:

    """
    ============================================================
    FEATURES DE COMPORTAMENTO - LOGS
    ============================================================

    Cria:

        - histórico
        - acumulados
        - médias históricas
        - variações
        - janelas de 2 e 3 meses
        - meses ativos
        - intensidade recente
        - indicadores de ausência de histórico

    As features utilizam somente informações disponíveis
    até a safra atual.
    """

    print(
        "  → Construindo features de comportamento dos logs..."
    )

    # ==========================================================
    # JANELA TEMPORAL
    # ==========================================================

    w = (
        Window
        .partitionBy(IDENTIFY)
        .orderBy("safra_date_dt")
    )

    # ==========================================================
    # 1. HISTÓRICO
    # ==========================================================

    df = df.withColumn(
        "meses_historico",
        F.row_number().over(w)
    )

    # ==========================================================
    # 2. ACUMULADOS
    # ==========================================================

    w_acum = (
        w.rowsBetween(
            Window.unboundedPreceding,
            0
        )
    )

    df = df.withColumn(
        "total_plays_acumulado",
        F.sum("total_plays").over(w_acum)
    )

    df = df.withColumn(
        "total_secs_acumulado",
        F.sum("total_secs").over(w_acum)
    )

    df = df.withColumn(
        "num_unq_acumulado",
        F.sum("num_unq").over(w_acum)
    )

    # ==========================================================
    # 3. MÉDIAS HISTÓRICAS
    # ==========================================================

    df = df.withColumn(
        "media_plays_mes_hist",
        F.when(
            F.col("meses_historico") > 0,
            F.col("total_plays_acumulado") /
            F.col("meses_historico")
        ).otherwise(0.0)
    )

    df = df.withColumn(
        "media_secs_mes_hist",
        F.when(
            F.col("meses_historico") > 0,
            F.col("total_secs_acumulado") /
            F.col("meses_historico")
        ).otherwise(0.0)
    )

    df = df.withColumn(
        "media_unq_mes_hist",
        F.when(
            F.col("meses_historico") > 0,
            F.col("num_unq_acumulado") /
            F.col("meses_historico")
        ).otherwise(0.0)
    )

    # ==========================================================
    # 4. VARIAÇÃO MENSAL
    # ==========================================================

    for col_name in [
        "total_plays",
        "total_secs",
        "num_unq"
    ]:

        lag_col = f"_{col_name}_lag1"

        # ------------------------------------------------------
        # Valor do mês anterior
        # ------------------------------------------------------

        df = df.withColumn(
            lag_col,
            F.lag(col_name).over(w)
        )

        # ------------------------------------------------------
        # Indicador de ausência de histórico
        # ------------------------------------------------------

        df = df.withColumn(
            f"sem_historico_{col_name}",
            F.when(
                F.col(lag_col).isNull(),
                1
            ).otherwise(0)
        )

        # ------------------------------------------------------
        # Indicador de mês anterior igual a zero
        # ------------------------------------------------------

        df = df.withColumn(
            f"anterior_zero_{col_name}",
            F.when(
                F.col(lag_col) == 0,
                1
            ).otherwise(0)
        )

        # ------------------------------------------------------
        # Variação percentual
        #
        # Só calculamos quando existe uma base positiva.
        # Nos demais casos usamos 0 e os indicadores acima
        # preservam a informação de que não havia base válida.
        # ------------------------------------------------------

        df = df.withColumn(
            f"var_{col_name}",
            F.when(
                F.col(lag_col) > 0,
                (
                    F.col(col_name) -
                    F.col(lag_col)
                ) / F.col(lag_col)
            ).otherwise(0.0)
        )

    # ==========================================================
    # 5. JANELAS RECENTES
    # ==========================================================

    for nome, deslocamento in [
        ("2m", 1),
        ("3m", 2)
    ]:

        w_roll = (
            Window
            .partitionBy(IDENTIFY)
            .orderBy("safra_date_dt")
            .rowsBetween(
                -deslocamento,
                0
            )
        )

        df = df.withColumn(
            f"total_plays_{nome}",
            F.sum("total_plays").over(w_roll)
        )

        df = df.withColumn(
            f"total_secs_{nome}",
            F.sum("total_secs").over(w_roll)
        )

        df = df.withColumn(
            f"num_unq_{nome}",
            F.sum("num_unq").over(w_roll)
        )

    # ==========================================================
    # 6. MESES COM ATIVIDADE
    # ==========================================================

    df = df.withColumn(
        "_mes_ativo",
        (
            F.col("total_plays") > 0
        ).cast("int")
    )

    for nome, deslocamento in [
        ("2m", 1),
        ("3m", 2)
    ]:

        w_roll = (
            Window
            .partitionBy(IDENTIFY)
            .orderBy("safra_date_dt")
            .rowsBetween(
                -deslocamento,
                0
            )
        )

        df = df.withColumn(
            f"meses_ativos_{nome}",
            F.sum("_mes_ativo").over(w_roll)
        )

    # ==========================================================
    # 7. INTENSIDADE RECENTE
    # ==========================================================

    df = df.withColumn(
        "media_plays_3m",
        F.col("total_plays_3m") / F.lit(3.0)
    )

    df = df.withColumn(
        "media_secs_3m",
        F.col("total_secs_3m") / F.lit(3.0)
    )

    df = df.withColumn(
        "media_unq_3m",
        F.col("num_unq_3m") / F.lit(3.0)
    )

    # ==========================================================
    # 8. LIMPEZA DAS COLUNAS AUXILIARES
    # ==========================================================

    cols_auxiliares = [
        c for c in df.columns
        if c.startswith("_")
    ]

    if cols_auxiliares:
        df = df.drop(*cols_auxiliares)

    return df

### Keep Público Alvo

In [ ]:
# ==============================================================
# 5. JOIN FINAL
# ==============================================================

def keep_logs_publico_alvo(
    df_logs_features: DataFrame,
    publico_alvo_path: str
) -> DataFrame:
    """
    ============================================================
    JOIN

    Para manter no logs somente quem é do público.
    """

    print("  → Associando logs ao público-alvo...")

    df_publico = spark.read.parquet(
        publico_alvo_path
    )


    # ----------------------------------------------------------
    # LEFT JOIN
    # ----------------------------------------------------------

    df_final = (
        df_publico.select(IDENTIFY, TIME_COL)
        .join(
            df_logs_features,
            on=[
                IDENTIFY,
                TIME_COL
            ],
            how="inner"
        )
    )

    return df_final

### Pipeline Main - LOGS

In [ ]:
%%time
# ==============================================================
# PIPELINE COMPLETO - LOGS
# ==============================================================

def run_pipeline_logs(
    input_path: str,
    publico_alvo_path: str,
    output_path: str,
    overwrite: bool = True
) -> DataFrame:

    print("=" * 70)
    print("PIPELINE DE FEATURE ENGINEERING - LOGS")
    print("=" * 70)

    # ==========================================================
    # 1. LEITURA
    # ==========================================================

    print("\n[1/7] Lendo logs...")

    df = spark.read.parquet(input_path)

    qtd_bruta = df.count()

    print(
        f"Logs brutos: {qtd_bruta:,}"
    )

    # ==========================================================
    # 2. PRÉ-PROCESSAMENTO
    # ==========================================================

    print("\n[2/7] Pré-processamento...")

    df = preprocess_logs(df)

    qtd_preprocessado = df.count()

    print(
        f"Logs após pré-processamento: "
        f"{qtd_preprocessado:,}"
    )

    # ==========================================================
    # 3. FILTRO DE HISTÓRICO CONTÍNUO
    # ==========================================================

    print(
        "\n[3/7] Filtrando clientes com histórico contínuo..."
    )

    df = filtrar_historico_continuo(df)

    qtd_continuo = df.count()

    print(
        f"Logs após filtro de continuidade: "
        f"{qtd_continuo:,}"
    )

    # ==========================================================
    # 4. AGREGAÇÃO MENSAL
    # ==========================================================

    print(
        "\n[4/7] Agregando logs mensalmente..."
    )

    df = aggregate_logs_monthly(df)

    qtd_mensal = df.count()

    print(
        f"Cliente × safra após agregação: "
        f"{qtd_mensal:,}"
    )

    # ==========================================================
    # 5. FEATURES DE COMPORTAMENTO
    # ==========================================================

    print(
        "\n[5/7] Criando features de comportamento..."
    )

    df = build_behavior_features_logs(df)

    qtd_features = df.count()

    print(
        f"Registros após feature engineering: "
        f"{qtd_features:,}"
    )

    # ==========================================================
    # 6. JOIN COM PÚBLICO-ALVO
    # ==========================================================

    print(
        "\n[6/7] Associando ao público-alvo..."
    )

    df_final = keep_logs_publico_alvo(
        df_logs_features=df,
        publico_alvo_path=publico_alvo_path
    )

    # ==========================================================
    # ORDENAÇÃO
    # ==========================================================

    df_final = df_final.orderBy(
        IDENTIFY,
        "safra_date_dt"
    )

    # ==========================================================
    # VALIDAÇÃO FINAL
    # ==========================================================

    print(
        "\n  → Validando dataset final..."
    )

    # ----------------------------------------------------------
    # Quantidade de registros
    # ----------------------------------------------------------

    qtd_final = df_final.count()

    # ----------------------------------------------------------
    # Quantidade de clientes
    # ----------------------------------------------------------

    qtd_clientes = (
        df_final
        .select(IDENTIFY)
        .distinct()
        .count()
    )

    # ----------------------------------------------------------
    # Quantidade de colunas
    # ----------------------------------------------------------

    qtd_colunas = len(df_final.columns)

    # ----------------------------------------------------------
    # NULLs
    # ----------------------------------------------------------

    print(
        "\n  → Verificando NULLs..."
    )

    null_exprs = [
        F.sum(
            F.col(c).isNull().cast("int")
        ).alias(c)
        for c in df_final.columns
    ]

    null_row = (
        df_final
        .agg(*null_exprs)
        .collect()[0]
    )

    nulls_encontrados = []

    for col_name in df_final.columns:

        qtd_null = null_row[col_name]

        if qtd_null > 0:

            nulls_encontrados.append(
                (
                    col_name,
                    qtd_null
                )
            )

    if nulls_encontrados:

        print(
            "    ⚠️ NULLs encontrados:"
        )

        for col_name, qtd_null in nulls_encontrados:

            print(
                f"       {col_name:<35}"
                f"{qtd_null:,}"
            )

    else:

        print(
            "    ✅ Nenhum NULL encontrado."
        )

    # ==========================================================
    # 7. SALVAMENTO
    # ==========================================================

    print(
        "\n[7/7] Salvando dataset..."
    )

    if overwrite:

        shutil.rmtree(
            output_path,
            ignore_errors=True
        )

    (
        df_final
        .write
        .mode("overwrite")
        .parquet(output_path)
    )

    # ==========================================================
    # RESUMO FINAL
    # ==========================================================

    print("\n" + "=" * 70)
    print("✅ PIPELINE DE LOGS CONCLUÍDO")
    print("=" * 70)

    print(
        f"Linhas brutas       : {qtd_bruta:,}"
    )

    print(
        f"Linhas pré-process. : {qtd_preprocessado:,}"
    )

    print(
        f"Linhas contínuas    : {qtd_continuo:,}"
    )

    print(
        f"Cliente × safra     : {qtd_mensal:,}"
    )

    print(
        f"Linhas finais       : {qtd_final:,}"
    )

    print(
        f"Clientes finais     : {qtd_clientes:,}"
    )

    print(
        f"Colunas finais      : {qtd_colunas:,}"
    )

    print(
        f"\nDataset salvo em:\n"
        f"{output_path}"
    )

    return df_final

input_logs = (
    "/content/drive/MyDrive/SANTANDER/"
    "user_logs.parquet"
)

publico_alvo_path = (
    "/content/drive/MyDrive/SANTANDER/features_members.parquet"
)

output_logs = (
    "/content/drive/MyDrive/SANTANDER/"
    "features_logs.parquet"
)


df_logs_final = run_pipeline_logs(
    input_path=input_logs,
    publico_alvo_path=publico_alvo_path,
    output_path=output_logs,
    overwrite=True
)

PIPELINE DE FEATURE ENGINEERING - LOGS

[1/7] Lendo logs...
Logs brutos: 26,758,971

[2/7] Pré-processamento...
ETAPA 1 - PRÉ-PROCESSAMENTO - LOGS
[PRE-PROCESSAMENTO] Registros iniciais: 26,758,971
[PRE-PROCESSAMENTO] Registros após validação temporal: 26,758,971
[PRE-PROCESSAMENTO] Registros após filtro de clientes presentes nas três bases: 14,841,572
[PRE-PROCESSAMENTO] Registros finais da etapa: 14,841,572
Logs após pré-processamento: 14,841,572

[3/7] Filtrando clientes com histórico contínuo...
[PRE-PROCESSAMENTO] Verificando continuidade das safras...
  Clientes antes       : 790,075
  Clientes depois      : 605,726
  Clientes removidos   : 184,349
  % removido            : 23.33%
Logs após filtro de continuidade: 11,697,565

[4/7] Agregando logs mensalmente...
  → Agregando logs por cliente e safra...
Cliente × safra após agregação: 11,697,565

[5/7] Criando features de comportamento...
  → Construindo features de comportamento dos logs...
Registros após feature engineering: 11,

# Transactions

### Consistência

In [ ]:
def preprocess_transactions(df: DataFrame) -> DataFrame:

    print("  → Iniciando pré-processamento de transactions...")

    # ==========================================================
    # 1. IDENTIFICADOR
    # ==========================================================

    if IDENTIFY not in df.columns:
        raise ValueError(
            f"Coluna '{IDENTIFY}' não encontrada."
        )

    df = df.withColumn(
        IDENTIFY,
        F.trim(
            F.col(IDENTIFY).cast("string")
        )
    )

    df = df.filter(
        F.col(IDENTIFY).isNotNull() &
        (F.col(IDENTIFY) != "")
    )

    print(
        f"  → Transactions após validação do identificador: "
        f"{df.count():,}"
    )

    # ==========================================================
    # 2. FILTRO DO UNIVERSO COMUM DAS TRÊS BASES
    #
    # O arquivo merged_ids_analysis.parquet foi criado
    # previamente através da análise de interseção das bases.
    #
    # presenca = 111
    #
    # significa:
    #
    #     1 -> presente em Members
    #     1 -> presente em Transactions
    #     1 -> presente em Logs
    #
    # Portanto, mantemos somente clientes presentes nas três
    # bases antes de executar as etapas mais pesadas de
    # processamento e feature engineering.
    # ==========================================================

    ids_validos = (
        spark.read.parquet(
            output_analysis_path
        )
        .select(IDENTIFY)
        .distinct()
    )

    df = df.join(
        ids_validos,
        on=IDENTIFY,
        how="inner"
    )

    print(
        f"  → Transactions após filtro de clientes "
        f"presentes nas três bases: "
        f"{df.count():,}"
    )

    # ==========================================================
    # 3. TRANSACTION DATE
    # ==========================================================

    if "transaction_date" not in df.columns:
        raise ValueError(
            "Coluna 'transaction_date' não encontrada."
        )

    transaction_raw = F.trim(
        F.col("transaction_date").cast("string")
    )

    df = df.withColumn(
        "transaction_date",
        F.coalesce(
            F.to_date(
                transaction_raw,
                "yyyyMMdd"
            ),
            F.to_date(
                transaction_raw,
                "yyyy-MM-dd"
            )
        )
    )

    df = df.filter(
        F.col("transaction_date").isNotNull()
    )

    # ==========================================================
    # 4. SAFRA
    # ==========================================================

    df = df.withColumn(
        "safra",
        F.date_format(
            F.col("transaction_date"),
            "yyyyMM"
        )
    )

    df = df.withColumn(
        "safra_date_dt",
        F.to_date(
            F.concat(
                F.col("safra"),
                F.lit("01")
            ),
            "yyyyMMdd"
        )
    )

    # ==========================================================
    # 5. MEMBERSHIP EXPIRE DATE
    # ==========================================================

    if "membership_expire_date" in df.columns:

        expire_raw = F.trim(
            F.col("membership_expire_date").cast("string")
        )

        df = df.withColumn(
            "membership_expire_date",
            F.coalesce(
                F.to_date(
                    expire_raw,
                    "yyyyMMdd"
                ),
                F.to_date(
                    expire_raw,
                    "yyyy-MM-dd"
                )
            )
        )

    # ==========================================================
    # 6. TIPAGEM
    # ==========================================================

    numeric_cols = {
        "payment_method_id": "int",
        "payment_plan_days": "int",
        "plan_list_price": "double",
        "actual_amount_paid": "double",
        "is_auto_renew": "int",
        "is_cancel": "int"
    }

    for col_name, dtype in numeric_cols.items():

        if col_name in df.columns:

            df = df.withColumn(
                col_name,
                F.col(col_name).cast(dtype)
            )

    # ==========================================================
    # 7. FLAGS DE QUALIDADE
    # ==========================================================

    # Indica ausência do preço de lista
    df = df.withColumn(
        "flag_plan_list_price_ausente",
        F.col("plan_list_price").isNull().cast("int")
    )

    # Indica ausência do valor efetivamente pago
    df = df.withColumn(
        "flag_actual_amount_paid_ausente",
        F.col("actual_amount_paid").isNull().cast("int")
    )

    # ==========================================================
    # 8. VALORES FINANCEIROS INVÁLIDOS
    # ==========================================================

    if "plan_list_price" in df.columns:

        df = df.withColumn(
            "plan_list_price",
            F.when(
                F.col("plan_list_price") < 0,
                None
            ).otherwise(
                F.col("plan_list_price")
            )
        )

    if "actual_amount_paid" in df.columns:

        df = df.withColumn(
            "actual_amount_paid",
            F.when(
                F.col("actual_amount_paid") < 0,
                None
            ).otherwise(
                F.col("actual_amount_paid")
            )
        )

    # ==========================================================
    # 9. FLAGS
    # ==========================================================

    if "is_auto_renew" in df.columns:

        df = df.withColumn(
            "is_auto_renew",
            F.when(
                F.col("is_auto_renew").isin(0, 1),
                F.col("is_auto_renew")
            ).otherwise(None)
        )

    if "is_cancel" in df.columns:

        df = df.withColumn(
            "is_cancel",
            F.when(
                F.col("is_cancel").isin(0, 1),
                F.col("is_cancel")
            ).otherwise(None)
        )

    # ==========================================================
    # 10. DIAGNÓSTICO
    # ==========================================================

    print("  → Período das transactions:")

    df.select(
        F.min("transaction_date").alias("min_date"),
        F.max("transaction_date").alias("max_date"),
        F.min("safra").alias("min_safra"),
        F.max("safra").alias("max_safra")
    ).show(truncate=False)

    print(
        f"  → Transactions após preprocessamento: "
        f"{df.count():,}"
    )

    return df

### Features

In [ ]:
def aggregate_transactions_monthly(
    df: DataFrame
) -> DataFrame:

    print("  → Agregando transactions por cliente + safra...")

    # ==========================================================
    # 1. AGREGAÇÃO MENSAL
    #    Unidade: 1 cliente + 1 safra
    # ==========================================================

    df_month = (
        df
        .groupBy(
            IDENTIFY,
            "safra",
            "safra_date_dt"
        )
        .agg(

            # ==================================================
            # ÚLTIMO ESTADO DO MÊS
            # ==================================================

            F.max_by(
                "payment_method_id",
                "transaction_date"
            ).alias("payment_method_id"),

            F.max_by(
                "payment_plan_days",
                "transaction_date"
            ).alias("payment_plan_days"),

            F.max_by(
                "plan_list_price",
                "transaction_date"
            ).alias("plan_list_price"),

            F.max_by(
                "actual_amount_paid",
                "transaction_date"
            ).alias("actual_amount_paid"),

            F.max_by(
                "is_auto_renew",
                "transaction_date"
            ).alias("is_auto_renew"),

            F.max_by(
                "membership_expire_date",
                "transaction_date"
            ).alias("membership_expire_date"),

            F.max_by(
                "is_cancel",
                "transaction_date"
            ).alias("is_cancel"),

            # ==================================================
            # QUALIDADE DOS DADOS
            # ==================================================

            F.max(
                "flag_plan_list_price_ausente"
            ).alias(
                "flag_plan_list_price_ausente"
            ),

            # ==================================================
            # MÉTRICAS FINANCEIRAS
            # ==================================================

            F.sum(
                F.coalesce(
                    F.col("actual_amount_paid"),
                    F.lit(0.0)
                )
            ).alias(
                "receita_mes"
            ),

            F.count("*").alias(
                "qtd_transacoes"
            ),

            F.avg(
                "actual_amount_paid"
            ).alias(
                "ticket_medio"
            ),

            # ==================================================
            # CANCELAMENTOS
            # ==================================================

            F.sum(
                F.coalesce(
                    F.col("is_cancel"),
                    F.lit(0)
                )
            ).alias(
                "cancelamentos_mes"
            )
        )
    )

    # ==========================================================
    # 2. JANELA TEMPORAL POR CLIENTE
    # ==========================================================

    w = (
        Window
        .partitionBy(IDENTIFY)
        .orderBy("safra_date_dt")
    )

    # ==========================================================
    # 3. VALORES DO MÊS ANTERIOR
    #
    # Criamos todos os LAGs uma única vez.
    # ==========================================================

    df_month = (
        df_month
        .withColumn(
            "_receita_prev",
            F.lag("receita_mes").over(w)
        )
        .withColumn(
            "_plan_list_price_prev",
            F.lag("plan_list_price").over(w)
        )
        .withColumn(
            "_data_prev",
            F.lag("safra_date_dt").over(w)
        )
    )

    # ==========================================================
    # 4. FLAG DE PRIMEIRA TRANSAÇÃO
    #
    # A ausência de _data_prev identifica a primeira observação
    # daquele cliente na série histórica.
    # ==========================================================

    df_month = df_month.withColumn(
        "flag_primeira_transacao",
        F.when(
            F.col("_data_prev").isNull(),
            1
        ).otherwise(0)
    )

    # ==========================================================
    # 5. FLAG DE PROMOÇÃO
    #
    # Promoção:
    # preço de tabela = 0
    # e valor efetivamente pago = 0
    # ==========================================================

    df_month = df_month.withColumn(
        "flag_promocao",
        F.when(
            (F.col("plan_list_price") == 0) &
            (F.col("actual_amount_paid") == 0),
            1
        ).otherwise(0)
    )

    # ==========================================================
    # 6. DELTA DE RECEITA
    #
    # Receita atual - receita anterior.
    #
    # Primeira observação:
    # NULL → 0
    #
    # O contexto é preservado por:
    # flag_primeira_transacao
    # ==========================================================

    df_month = df_month.withColumn(
        "delta_receita",
        F.when(
            F.col("_receita_prev").isNull(),
            F.lit(0.0)
        ).otherwise(
            F.col("receita_mes") -
            F.col("_receita_prev")
        )
    )

    # ==========================================================
    # 7. ENTRADA EM RECEITA
    #
    # Cliente passa de receita zero/ausente para receita positiva.
    # ==========================================================

    df_month = df_month.withColumn(
        "entrou_em_receita",
        F.when(
            (F.coalesce(
                F.col("_receita_prev"),
                F.lit(0.0)
            ) <= 0) &
            (F.col("receita_mes") > 0),
            1
        ).otherwise(0)
    )

    # ==========================================================
    # 8. SAÍDA DE RECEITA
    #
    # Cliente passa de receita positiva para receita zero.
    # ==========================================================

    df_month = df_month.withColumn(
        "saiu_de_receita",
        F.when(
            (F.col("_receita_prev") > 0) &
            (F.col("receita_mes") <= 0),
            1
        ).otherwise(0)
    )

    # ==========================================================
    # 9. VARIAÇÃO PERCENTUAL DA RECEITA
    #
    # Fórmula:
    #
    # (receita_atual - receita_anterior) /
    # receita_anterior
    #
    # Casos não calculáveis:
    #
    # 1. Não existe receita anterior
    # 2. Receita anterior = 0
    #
    # Nesses casos:
    # NULL → 0
    #
    # O contexto é preservado pelas flags:
    # - flag_primeira_transacao
    # - entrou_em_receita
    # - saiu_de_receita
    # ==========================================================

    df_month = df_month.withColumn(
        "var_receita",
        F.when(
            F.col("_receita_prev").isNull(),
            F.lit(0.0)
        )
        .when(
            F.col("_receita_prev") == 0,
            F.lit(0.0)
        )
        .otherwise(
            (
                F.col("receita_mes") -
                F.col("_receita_prev")
            ) / F.col("_receita_prev")
        )
    )

    # ==========================================================
    # 10. MUDANÇA DE PREÇO
    #
    # -1 = diminuiu
    #  0 = não mudou
    #  1 = aumentou
    #
    # Primeira observação:
    # NULL → 0
    #
    # O contexto é preservado por:
    # flag_primeira_transacao
    # ==========================================================

    df_month = df_month.withColumn(
        "mudanca_preco",
        F.when(
            F.col("_plan_list_price_prev").isNull(),
            0
        )
        .when(
            F.col("plan_list_price") >
            F.col("_plan_list_price_prev"),
            1
        )
        .when(
            F.col("plan_list_price") <
            F.col("_plan_list_price_prev"),
            -1
        )
        .otherwise(0)
    )

    # ==========================================================
    # 11. FLAGS DE MUDANÇA DE PREÇO
    # ==========================================================

    df_month = (
        df_month
        .withColumn(
            "flag_aumento_preco",
            F.when(
                F.col("mudanca_preco") == 1,
                1
            ).otherwise(0)
        )
        .withColumn(
            "flag_reducao_preco",
            F.when(
                F.col("mudanca_preco") == -1,
                1
            ).otherwise(0)
        )
    )

    # ==========================================================
    # 12. INTERVALO ENTRE TRANSAÇÕES
    #
    # Diferença em meses entre a safra atual e a anterior.
    #
    # Primeira observação:
    # NULL → 0
    #
    # O contexto é preservado por:
    # flag_primeira_transacao
    # ==========================================================

    df_month = df_month.withColumn(
        "intervalo_transacoes_meses",
        F.when(
            F.col("_data_prev").isNull(),
            F.lit(0.0)
        )
        .otherwise(
            F.months_between(
                F.col("safra_date_dt"),
                F.col("_data_prev")
            )
        )
    )

    # ==========================================================
    # 13. FLAG DE GAP ENTRE TRANSAÇÕES
    #
    # Gap quando o intervalo é maior que 1 mês.
    # ==========================================================

    df_month = df_month.withColumn(
        "flag_gap_transacoes",
        F.when(
            F.col("intervalo_transacoes_meses") > 1,
            1
        ).otherwise(0)
    )

    # ==========================================================
    # 14. RECEITA ACUMULADA
    # ==========================================================

    df_month = df_month.withColumn(
        "receita_acumulada",
        F.sum("receita_mes").over(
            w.rowsBetween(
                Window.unboundedPreceding,
                0
            )
        )
    )

    # ==========================================================
    # 15. RECEITA RECENTE
    #
    # 1m = mês atual
    # 2m = mês atual + observação anterior
    # 3m = mês atual + 2 observações anteriores
    #
    # Observação:
    # a janela é baseada em linhas observadas, não em calendário.
    # ==========================================================

    for nome, deslocamento in [
        ("1m", 0),
        ("2m", 1),
        ("3m", 2)
    ]:

        w_roll = (
            Window
            .partitionBy(IDENTIFY)
            .orderBy("safra_date_dt")
            .rowsBetween(
                -deslocamento,
                0
            )
        )

        df_month = (
            df_month
            .withColumn(
                f"receita_{nome}",
                F.sum("receita_mes").over(w_roll)
            )
            .withColumn(
                f"cancelamentos_{nome}",
                F.sum("cancelamentos_mes").over(w_roll)
            )
        )

    # ==========================================================
    # 16. FLAGS DE CANCELAMENTO RECENTE
    # ==========================================================

    for nome in ["1m", "2m", "3m"]:

        df_month = df_month.withColumn(
            f"flag_cancelamento_{nome}",
            (
                F.coalesce(
                    F.col(f"cancelamentos_{nome}"),
                    F.lit(0)
                ) > 0
            ).cast("int")
        )

    # ==========================================================
    # 17. HISTÓRICO DE AUTO-RENEW
    # ==========================================================

    df_month = df_month.withColumn(
        "taxa_auto_renew_hist",
        F.avg("is_auto_renew").over(
            w.rowsBetween(
                Window.unboundedPreceding,
                0
            )
        )
    )

    # ==========================================================
    # 18. GARANTIA DAS FEATURES ESTRUTURAIS
    #
    # Aqui NÃO estamos mascarando qualquer problema do dataset.
    #
    # Estamos tratando especificamente os NULLs esperados
    # dessas features:
    #
    # - primeira observação
    # - ausência de receita anterior
    # - receita anterior igual a zero
    #
    # As flags acima preservam o significado desses casos.
    # ==========================================================

    df_month = (
        df_month
        .withColumn(
            "delta_receita",
            F.coalesce(
                F.col("delta_receita"),
                F.lit(0.0)
            )
        )
        .withColumn(
            "var_receita",
            F.coalesce(
                F.col("var_receita"),
                F.lit(0.0)
            )
        )
        .withColumn(
            "intervalo_transacoes_meses",
            F.coalesce(
                F.col("intervalo_transacoes_meses"),
                F.lit(0.0)
            )
        )
    )

    # ==========================================================
    # 19. REMOVER COLUNAS AUXILIARES
    # ==========================================================

    df_month = df_month.drop(
        "_receita_prev",
        "_plan_list_price_prev",
        "_data_prev"
    )

    return df_month

### Behavior

In [ ]:
def build_behavior_features_transactions(
    df: DataFrame
) -> DataFrame:

    print("  → Construindo features de comportamento...")

    # ==========================================================
    # JANELA TEMPORAL
    # ==========================================================

    w = (
        Window
        .partitionBy(IDENTIFY)
        .orderBy("safra_date_dt")
    )

    # ==========================================================
    # 1. INFORMAÇÕES DO PERÍODO ANTERIOR
    # ==========================================================

    df = (
        df
        .withColumn(
            "_receita_prev",
            F.lag("receita_mes").over(w)
        )
        .withColumn(
            "_observacao_prev",
            F.lag(F.lit(1)).over(w)
        )
        .withColumn(
            "_data_prev",
            F.lag("safra_date_dt").over(w)
        )
        .withColumn(
            "_plan_prev",
            F.lag("payment_plan_days").over(w)
        )
        .withColumn(
            "_auto_renew_prev",
            F.lag("is_auto_renew").over(w)
        )
    )

    # ==========================================================
    # 2. EXISTE OBSERVAÇÃO ANTERIOR?
    # ==========================================================

    df = df.withColumn(
        "tem_observacao_anterior",
        F.when(
            F.col("_observacao_prev").isNotNull(),
            1
        ).otherwise(0)
    )

    # ==========================================================
    # 3. EXISTE RECEITA ANTERIOR?
    #
    # receita_mes é construída com COALESCE no agregado,
    # portanto uma receita anterior NULL representa ausência
    # estrutural de observação anterior.
    # ==========================================================

    df = df.withColumn(
        "tem_receita_anterior",
        F.when(
            F.col("_receita_prev").isNotNull(),
            1
        ).otherwise(0)
    )

    # ==========================================================
    # 4. FLAG PRIMEIRA OBSERVAÇÃO
    #
    # Não existe período anterior para este cliente.
    # ==========================================================

    df = df.withColumn(
        "flag_primeira_observacao",
        F.when(
            F.col("_data_prev").isNull(),
            1
        ).otherwise(0)
    )

    # Mantemos também este nome, caso já esteja sendo utilizado
    # em outras etapas do projeto.
    df = df.withColumn(
        "flag_primeira_transacao",
        F.col("flag_primeira_observacao")
    )

    # ==========================================================
    # 5. DELTA DE RECEITA
    #
    # Receita atual - receita anterior.
    #
    # Primeira observação:
    # NULL → 0.0
    #
    # A informação de que não havia histórico é preservada
    # por flag_primeira_observacao.
    # ==========================================================

    df = df.withColumn(
        "delta_receita",
        F.when(
            F.col("_receita_prev").isNull(),
            F.lit(0.0)
        ).otherwise(
            F.col("receita_mes") -
            F.col("_receita_prev")
        )
    )

    # ==========================================================
    # 6. RECEITA ANTERIOR IGUAL A ZERO
    #
    # Importante para distinguir:
    #
    # NULL estrutural:
    # não existe histórico anterior
    #
    # de:
    #
    # receita anterior = 0
    # ==========================================================

    df = df.withColumn(
        "receita_anterior_zero",
        F.when(
            F.col("_receita_prev") == 0,
            1
        ).otherwise(0)
    )

    # ==========================================================
    # 7. ENTRADA EM RECEITA
    #
    # 0 → positivo
    #
    # A primeira observação não é considerada "entrada",
    # pois não sabemos o que ocorreu antes dela.
    # ==========================================================

    df = df.withColumn(
        "entrou_em_receita",
        F.when(
            (F.col("_receita_prev").isNotNull()) &
            (F.col("_receita_prev") <= 0) &
            (F.col("receita_mes") > 0),
            1
        ).otherwise(0)
    )

    # ==========================================================
    # 8. SAÍDA DE RECEITA
    #
    # positivo → 0
    # ==========================================================

    df = df.withColumn(
        "saiu_de_receita",
        F.when(
            (F.col("_receita_prev").isNotNull()) &
            (F.col("_receita_prev") > 0) &
            (F.col("receita_mes") <= 0),
            1
        ).otherwise(0)
    )

    # ==========================================================
    # 9. VARIAÇÃO PERCENTUAL DA RECEITA
    #
    # Fórmula:
    #
    # (atual - anterior) / anterior
    #
    # Existem dois casos em que a divisão não é calculável:
    #
    # 1. não existe receita anterior
    # 2. receita anterior = 0
    #
    # Ambos recebem 0.0.
    #
    # O contexto permanece nas flags:
    #
    # - flag_primeira_observacao
    # - receita_anterior_zero
    # - entrou_em_receita
    # - saiu_de_receita
    # ==========================================================

    df = df.withColumn(
        "var_receita",
        F.when(
            F.col("_receita_prev").isNull(),
            F.lit(0.0)
        )
        .when(
            F.col("_receita_prev") == 0,
            F.lit(0.0)
        )
        .otherwise(
            (
                F.col("receita_mes") -
                F.col("_receita_prev")
            ) / F.col("_receita_prev")
        )
    )

    # ==========================================================
    # 10. MUDANÇA DE PLANO
    # ==========================================================

    df = df.withColumn(
        "mudou_plano",
        F.when(
            F.col("_plan_prev").isNotNull() &
            F.col("payment_plan_days").isNotNull() &
            (
                F.col("payment_plan_days") !=
                F.col("_plan_prev")
            ),
            1
        ).otherwise(0)
    )

    # ==========================================================
    # 11. UPGRADE DE PLANO
    # ==========================================================

    df = df.withColumn(
        "upgrade_plano",
        F.when(
            F.col("_plan_prev").isNotNull() &
            F.col("payment_plan_days").isNotNull() &
            (
                F.col("payment_plan_days") >
                F.col("_plan_prev")
            ),
            1
        ).otherwise(0)
    )

    # ==========================================================
    # 12. DOWNGRADE DE PLANO
    # ==========================================================

    df = df.withColumn(
        "downgrade_plano",
        F.when(
            F.col("_plan_prev").isNotNull() &
            F.col("payment_plan_days").isNotNull() &
            (
                F.col("payment_plan_days") <
                F.col("_plan_prev")
            ),
            1
        ).otherwise(0)
    )

    # ==========================================================
    # 13. RECEITA ACUMULADA
    # ==========================================================

    df = df.withColumn(
        "receita_acumulada",
        F.coalesce(
            F.sum("receita_mes").over(
                w.rowsBetween(
                    Window.unboundedPreceding,
                    Window.currentRow
                )
            ),
            F.lit(0.0)
        )
    )

    # ==========================================================
    # 14. INTERVALO ENTRE TRANSAÇÕES
    #
    # Primeira observação:
    # NULL → 0
    #
    # Mantemos flag_primeira_observacao = 1 para identificar
    # que o zero representa ausência de histórico anterior.
    # ==========================================================

    df = df.withColumn(
        "intervalo_transacoes_meses",
        F.when(
            F.col("_data_prev").isNull(),
            F.lit(0.0)
        )
        .otherwise(
            F.floor(
                F.months_between(
                    F.col("safra_date_dt"),
                    F.col("_data_prev")
                )
            ).cast("double")
        )
    )

    # ==========================================================
    # 15. GAP NO HISTÓRICO
    #
    # Intervalo maior que 1 mês.
    # ==========================================================

    df = df.withColumn(
        "flag_gap_transacoes",
        F.when(
            F.col("intervalo_transacoes_meses") > 1,
            1
        ).otherwise(0)
    )

    # ==========================================================
    # 16. JANELAS RECENTES
    #
    # 1m = observação atual
    # 2m = atual + anterior
    # 3m = atual + duas anteriores
    #
    # Atenção:
    # rowsBetween é baseado em linhas observadas.
    # ==========================================================

    for nome, deslocamento in [
        ("1m", 0),
        ("2m", 1),
        ("3m", 2)
    ]:

        w_roll = (
            Window
            .partitionBy(IDENTIFY)
            .orderBy("safra_date_dt")
            .rowsBetween(
                -deslocamento,
                Window.currentRow
            )
        )

        df = (
            df
            .withColumn(
                f"receita_{nome}",
                F.coalesce(
                    F.sum("receita_mes").over(w_roll),
                    F.lit(0.0)
                )
            )
            .withColumn(
                f"cancelamentos_{nome}",
                F.coalesce(
                    F.sum("cancelamentos_mes").over(w_roll),
                    F.lit(0.0)
                )
            )
            .withColumn(
                f"qtd_transacoes_{nome}",
                F.coalesce(
                    F.sum("qtd_transacoes").over(w_roll),
                    F.lit(0.0)
                )
            )
    )

    # ==========================================================
    # 17. FLAGS DE CANCELAMENTO
    # ==========================================================

    for nome in ["1m", "2m", "3m"]:

        df = df.withColumn(
            f"flag_cancelamento_{nome}",
            F.when(
                F.coalesce(
                    F.col(f"cancelamentos_{nome}"),
                    F.lit(0.0)
                ) > 0,
                1
            ).otherwise(0)
        )

    # ==========================================================
    # 18. TAXA HISTÓRICA DE AUTO-RENEW
    # ==========================================================

    df = df.withColumn(
        "taxa_auto_renew_hist",
        F.coalesce(
            F.avg("is_auto_renew").over(
                w.rowsBetween(
                    Window.unboundedPreceding,
                    Window.currentRow
                )
            ),
            F.lit(0.0)
        )
    )

    # ==========================================================
    # 19. MUDANÇA DE AUTO-RENEW
    # ==========================================================

    df = df.withColumn(
        "mudou_auto_renew",
        F.when(
            F.col("_auto_renew_prev").isNotNull() &
            F.col("is_auto_renew").isNotNull() &
            (
                F.col("_auto_renew_prev") !=
                F.col("is_auto_renew")
            ),
            1
        ).otherwise(0)
    )

    # ==========================================================
    # 20. DESATIVAÇÃO DE AUTO-RENEW
    # ==========================================================

    df = df.withColumn(
        "desativou_auto_renew",
        F.when(
            (F.col("_auto_renew_prev") == 1) &
            (F.col("is_auto_renew") == 0),
            1
        ).otherwise(0)
    )

    # ==========================================================
    # 21. GARANTIA FINAL DAS FEATURES TRATADAS
    #
    # Essas são features cujo NULL é estruturalmente esperado.
    # ==========================================================

    df = (
        df
        .withColumn(
            "delta_receita",
            F.coalesce(
                F.col("delta_receita"),
                F.lit(0.0)
            )
        )
        .withColumn(
            "var_receita",
            F.coalesce(
                F.col("var_receita"),
                F.lit(0.0)
            )
        )
        .withColumn(
            "intervalo_transacoes_meses",
            F.coalesce(
                F.col("intervalo_transacoes_meses"),
                F.lit(0.0)
            )
        )
    )

    # ==========================================================
    # 22. REMOVER COLUNAS AUXILIARES
    # ==========================================================

    df = df.drop(
        "_receita_prev",
        "_observacao_prev",
        "_data_prev",
        "_plan_prev",
        "_auto_renew_prev"
    )

    return df

### Público Alvo

In [ ]:
def filter_transactions_to_publico_alvo(
    df_transactions: DataFrame,
    publico_alvo_path: str
) -> DataFrame:

    print("  → Filtrando transactions para o público-alvo...")

    df_publico = (
        spark.read.parquet(publico_alvo_path)
        .select(
            IDENTIFY,
            "safra"
        )
        .withColumn(
            IDENTIFY,
            F.trim(F.col(IDENTIFY).cast("string"))
        )
        .withColumn(
            "safra",
            F.col("safra").cast("string")
        )
        .distinct()
    )

    df_transactions = (
        df_transactions
        .withColumn(
            "safra",
            F.col("safra").cast("string")
        )
    )

    df_transactions = (
        df_transactions
        .join(
            df_publico,
            on=[
                IDENTIFY,
                "safra"
            ],
            how="inner"
        )
    )

    print(
        f"  → Transactions após filtro: "
        f"{df_transactions.count():,}"
    )

    return df_transactions

### Pipeline Main - TRANSACTIONS

In [ ]:
%%time
def run_pipeline_transactions(
    input_path,
    publico_alvo_path,
    output_path,
    overwrite=True
):

    print("=" * 70)
    print("PIPELINE DE FEATURE ENGINEERING - TRANSACTIONS")
    print("=" * 70)

    # ==========================================================
    # 1. LEITURA
    # ==========================================================

    print("\n[1/6] Lendo transactions...")

    df = spark.read.parquet(input_path)

    print(
        f"Transactions brutas: {df.count():,}"
    )

    # ==========================================================
    # 2. PRÉ-PROCESSAMENTO
    # ==========================================================

    print("\n[2/6] Pré-processamento...")

    df = preprocess_transactions(df)

    # ==========================================================
    # 3. AGREGAÇÃO MENSAL
    # ==========================================================

    print("\n[3/6] Agregando transactions mensalmente...")

    df_month = aggregate_transactions_monthly(df)

    print(
        f"Linhas após agregação mensal: "
        f"{df_month.count():,}"
    )

    # ==========================================================
    # 4. FEATURE ENGINEERING
    # ==========================================================

    print("\n[4/6] Construindo features temporais...")

    df_features = build_behavior_features_transactions(
        df_month
    )

    # ==========================================================
    # 5. PÚBLICO-ALVO
    # ==========================================================

    print("\n[5/6] Carregando público-alvo...")

    df_publico = (
        spark.read.parquet(publico_alvo_path)
        .select(
            IDENTIFY,
            "safra"
        )
        .withColumn(
            IDENTIFY,
            F.trim(
                F.col(IDENTIFY).cast("string")
            )
        )
        .withColumn(
            "safra",
            F.col("safra").cast("string")
        )
        .distinct()
    )


    # ==========================================================
    # 6. JOIN
    # ==========================================================

    print("\n[6/6] Associando features ao público-alvo...")

    df_final = (
        df_publico
        .join(
            df_features,
            on=[
                IDENTIFY,
                "safra"
            ],
            how="inner"
        )
        .orderBy(
            IDENTIFY,
            "safra"
        )
    )

    # ==========================================================
    # DIAGNÓSTICO DE NULLS
    # ==========================================================

    print("\n" + "=" * 70)
    print("DIAGNÓSTICO DE NULLS")
    print("=" * 70)

    null_cols = [
        "plan_list_price",
        "razao_preco_pago",
        "var_receita",
        "intervalo_transacoes_meses",
        "delta_receita",
        "ticket_medio",
        "membership_expire_date",
        "is_auto_renew"
    ]

    for col_name in null_cols:

        if col_name in df_final.columns:

            qtd_null = (
                df_final
                .filter(F.col(col_name).isNull())
                .count()
            )

            print(
                f"{col_name:35s}: "
                f"{qtd_null:,}"
            )


    # ==========================================================
    # SALVAMENTO
    # ==========================================================

    print("\nSalvando dataset...")

    if overwrite:
        shutil.rmtree(
            output_path,
            ignore_errors=True
        )

    (
        df_final
        .write
        .mode("overwrite")
        .parquet(output_path)
    )

    print("\n" + "=" * 70)
    print("✅ PIPELINE DE TRANSACTIONS CONCLUÍDO!")
    print("=" * 70)

    print(
        f"Linhas finais: {df_final.count():,}"
    )

    print(
        f"Colunas finais: {len(df_final.columns)}"
    )

    return df_final

input_path = (
    "/content/drive/MyDrive/SANTANDER/"
    "transactions.parquet"
)

publico_alvo_path = (
    "/content/drive/MyDrive/SANTANDER/"
    "features_members.parquet"
)

output_path = (
    "/content/drive/MyDrive/SANTANDER/"
    "features_transactions.parquet"
)

df_transactions = run_pipeline_transactions(
    input_path=input_path,
    publico_alvo_path=publico_alvo_path,
    output_path=output_path,
    overwrite=True
)

PIPELINE DE FEATURE ENGINEERING - TRANSACTIONS

[1/6] Lendo transactions...
Transactions brutas: 20,712,225

[2/6] Pré-processamento...
  → Iniciando pré-processamento de transactions...
  → Transactions após validação do identificador: 20,712,225
  → Transactions após filtro de clientes presentes nas três bases: 13,897,125
  → Período das transactions:
+----------+----------+---------+---------+
|min_date  |max_date  |min_safra|max_safra|
+----------+----------+---------+---------+
|2015-01-01|2017-02-28|201501   |201702   |
+----------+----------+---------+---------+

  → Transactions após preprocessamento: 13,897,125

[3/6] Agregando transactions mensalmente...
  → Agregando transactions por cliente + safra...
Linhas após agregação mensal: 13,897,125

[4/6] Construindo features temporais...
  → Construindo features de comportamento...

[5/6] Carregando público-alvo...

[6/6] Associando features ao público-alvo...

DIAGNÓSTICO DE NULLS
plan_list_price                    : 0
var_recei

In [ ]:
print("Checking for duplicate 'msno' in df_transactions_filtered:")
duplicate_msnos = (
    df_transactions
    .groupBy("msno")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_msnos.count() > 0:
    print("Found duplicate 'msno' values:")
    duplicate_msnos.show(truncate=False)
else:
    print("No duplicate 'msno' values found.")

Checking for duplicate 'msno' in df_transactions_filtered:
Found duplicate 'msno' values:
+--------------------------------------------+-----+
|msno                                        |count|
+--------------------------------------------+-----+
|+/namlXq+u3izRjHCFJV4MgqcXcLidZYszVsROOq/y4=|9    |
|+0/X9tkmyHyet9X80G6GTrDFHnJqvai8d1ZPhayT0os=|9    |
|+09YGn842g6h2EZUXe0VWeC4bBoCbDGfUboitc0vIHw=|9    |
|+0jTOa6KGPk1vtNTwRDMZc/McUo41AeuwV3ndo54Y+Q=|9    |
|+2Df04hr61UUJijMM2xR97gtoQWWDJpnJVKQ7VMYN9o=|9    |
|+2eLsQv6T46iKwO+m+r6OFI2X3Oc9dGBMdti2COAe4w=|9    |
|+2vC1rM36Emx77UanRb3cUohWiIf7knfVIDO2+R78BE=|9    |
|+A534cfk3ylNGsQ3d8UOkVvs8u2b7+UJqHODG4jfbSg=|9    |
|+C+D7ghIInae/G+6glbbsrslGxrxVPlr34lYCf0+lKs=|9    |
|+Ee3Z6oRlCNX6Kf4vKcWHfzCn3IwOj1dqflnyk9NYzA=|9    |
|+GVaRBskgRjfyLprhA1g/77D0kfnVYRUEBLY2DAVtUc=|9    |
|+MFM2WIg07EADKRI0gm/Vq+3s2jDW838pVt477htI8k=|9    |
|+Ue9TTtCJhYP0wa97yBH1bdBQ/8w0A/VKGL07aVPxDs=|9    |
|+Vzca1ck/BX03oBYaZ6itKg0TRbjlTguSvBSoklmQdM=|6    |
|+X+E2xPj

In [ ]:
msno_list = [
    "++4RuqBw0Ss6bQU4oMxaRlbBPoWzoEiIZaxPM04Y4+U=",
    "++OepqRK4wiYg4Chl+qqo7TrwM+i9KZc3Ez/Swbjjew=",
    "+0/X9tkmyHyet9X80G6GTrDFHnJqvai8d1ZPhayT0os="
]

# Load the original transactions.parquet file to find these msno's
df_original_transactions = spark.read.parquet("/content/drive/MyDrive/SANTANDER/transactions.parquet")

df_original_transactions.filter(F.col("msno").isin(msno_list)).orderBy("msno", "transaction_date").show(truncate=False)

+--------------------------------------------+-----------------+-----------------+---------------+------------------+-------------+----------------+----------------------+---------+------+
|msno                                        |payment_method_id|payment_plan_days|plan_list_price|actual_amount_paid|is_auto_renew|transaction_date|membership_expire_date|is_cancel|safra |
+--------------------------------------------+-----------------+-----------------+---------------+------------------+-------------+----------------+----------------------+---------+------+
|++4RuqBw0Ss6bQU4oMxaRlbBPoWzoEiIZaxPM04Y4+U=|41               |30               |129            |129               |1            |20150113        |20150213              |0        |201501|
|++4RuqBw0Ss6bQU4oMxaRlbBPoWzoEiIZaxPM04Y4+U=|41               |30               |129            |129               |1            |20150213        |20150313              |0        |201502|
|++4RuqBw0Ss6bQU4oMxaRlbBPoWzoEiIZaxPM04Y4+U=|41       

In [ ]:
df_transactions.filter(F.col("msno") == msno_list[2]).show(truncate=False)

+--------------------------------------------+------+-------------+-----------------+-----------------+---------------+------------------+-------------+----------------------+---------+----------------------------+-----------+--------------+------------+-----------------+-----------------------+-------------+-------------+-----------------+---------------+-----------+-------------+------------------+------------------+--------------------------+-------------------+-----------------+----------+----------------+----------+----------------+----------+----------------+--------------------+--------------------+--------------------+--------------------+-----------------------+--------------------+------------------------+---------------------+-----------+-------------+---------------+-----------------+-----------------+-----------------+----------------+--------------------+
|msno                                        |safra |safra_date_dt|payment_method_id|payment_plan_days|plan_list_price

# Master

In [ ]:
%%time
%%time

from pyspark.sql import functions as F


def join_datasets_features(
    members_path: str,
    logs_path: str,
    transactions_path: str
):

    print("=" * 70)
    print("JOIN FINAL - MEMBERS × LOGS × TRANSACTIONS")
    print("=" * 70)

    # ==========================================================
    # 1. LEITURA
    # ==========================================================

    print("\n[1] Lendo datasets...")

    df_members = (
        spark.read.parquet(members_path)
        .select(
            "*"
        )
        .withColumn(
            "msno",
            F.trim(F.col("msno").cast("string"))
        )
        .withColumn(
            "safra",
            F.col("safra").cast("string")
        )
        .filter(
            F.col("msno").isNotNull() &
            F.col("safra").isNotNull()
        )
    )

    df_logs = (
        spark.read.parquet(logs_path)
        .withColumn(
            "msno",
            F.trim(F.col("msno").cast("string"))
        )
        .withColumn(
            "safra",
            F.col("safra").cast("string")
        )
        .filter(
            F.col("msno").isNotNull() &
            F.col("safra").isNotNull()
        )
    )

    df_transactions = (
        spark.read.parquet(transactions_path)
        .withColumn(
            "msno",
            F.trim(F.col("msno").cast("string"))
        )
        .withColumn(
            "safra",
            F.col("safra").cast("string")
        )
        .filter(
            F.col("msno").isNotNull() &
            F.col("safra").isNotNull()
        )
    )

    # ==========================================================
    # 2. REMOVER COLUNAS REDUNDANTES
    # ==========================================================

    print("\n[2] Removendo colunas redundantes...")

    if "safra_date_dt" in df_logs.columns:
        df_logs = df_logs.drop("safra_date_dt")

    if "safra_date_dt" in df_transactions.columns:
        df_transactions = df_transactions.drop("safra_date_dt")

    # ==========================================================
    # 3. CRIAR SOMENTE AS CHAVES
    # ==========================================================

    print("\n[3] Criando conjuntos de chaves...")

    members_keys = df_members.select(
        "msno",
        "safra"
    )

    logs_keys = df_logs.select(
        "msno",
        "safra"
    )

    transactions_keys = df_transactions.select(
        "msno",
        "safra"
    )

    # ==========================================================
    # 4. INTERSEÇÃO DAS TRÊS BASES
    # ==========================================================

    print("\n[4] Calculando interseção das chaves...")

    chaves_finais = (
        members_keys
        .join(
            logs_keys,
            on=["msno", "safra"],
            how="inner"
        )
        .join(
            transactions_keys,
            on=["msno", "safra"],
            how="inner"
        )
    )

    # ==========================================================
    # 5. FILTRAR AS BASES PELAS CHAVES FINAIS
    # ==========================================================

    print("\n[5] Filtrando Members...")

    members_final = (
        df_members
        .join(
            chaves_finais,
            on=["msno", "safra"],
            how="left_semi"
        )
    )

    print("\n[6] Filtrando Logs...")

    logs_final = (
        df_logs
        .join(
            chaves_finais,
            on=["msno", "safra"],
            how="left_semi"
        )
    )

    print("\n[7] Filtrando Transactions...")

    transactions_final = (
        df_transactions
        .join(
            chaves_finais,
            on=["msno", "safra"],
            how="left_semi"
        )
    )

    # ==========================================================
    # 6. JOIN FINAL DAS FEATURES
    # ==========================================================

    print("\n[8] Construindo MASTER...")

    df_final = (
        members_final
        .join(
            logs_final,
            on=["msno", "safra"],
            how="inner"
        )
        .join(
            transactions_final,
            on=["msno", "safra"],
            how="inner"
        )
    )

    # ==========================================================
    # 7. FILTRO DE TEMPO
    # ==========================================================

    print(
        "\n[9] Aplicando "
        "tempo_cliente_meses >= 3..."
    )

    df_final = df_final.filter(
        F.col("tempo_cliente_meses") >= 3
    )

    print("\n" + "=" * 70)
    print("MASTER CONSTRUÍDA COM SUCESSO!")
    print("=" * 70)

    return df_final


members_path = (
    "/content/drive/MyDrive/SANTANDER/"
    "features_members.parquet"
)

logs_path = (
    "/content/drive/MyDrive/SANTANDER/"
    "features_logs.parquet"
)

transactions_path = (
    "/content/drive/MyDrive/SANTANDER/"
    "features_transactions.parquet"
)

output_path_final = (
    "/content/drive/MyDrive/SANTANDER/"
    "master.parquet"
)

dfs_final = join_datasets_features(
    members_path=members_path,
    logs_path=logs_path,
    transactions_path=transactions_path
)

dfs_final.write.mode("overwrite").parquet(output_path_final)
print(f"DataFrame saved to: {output_path_final}")

JOIN FINAL - MEMBERS × LOGS × TRANSACTIONS

[1] Lendo datasets...

[2] Removendo colunas redundantes...

[3] Criando conjuntos de chaves...

[4] Calculando interseção das chaves...

[5] Filtrando Members...

[6] Filtrando Logs...

[7] Filtrando Transactions...

[8] Construindo MASTER...

[9] Aplicando tempo_cliente_meses >= 3...

MASTER CONSTRUÍDA COM SUCESSO!
DataFrame saved to: /content/drive/MyDrive/SANTANDER/master.parquet
CPU times: user 139 ms, sys: 32.9 ms, total: 172 ms
Wall time: 6min 17s
CPU times: user 150 ms, sys: 32.9 ms, total: 183 ms
Wall time: 6min 17s


In [ ]:
%time
output_path =  "/content/drive/MyDrive/SANTANDER/master.parquet"

dfs_master = spark.read.parquet(output_path)
dfs_master = dfs_master.withColumn("ultima_safra_ativa", F.when(F.col("ultima_safra_ativa").isNull(), F.col("safra_date_dt")).otherwise(F.col("ultima_safra_ativa")))

print("Verifying null values in dfs_master:")
null_counts = []
for col_name in dfs_master.columns:
    null_count = dfs_master.filter(F.col(col_name).isNull()).count()
    if null_count > 0:
        null_counts.append((col_name, null_count))

if len(null_counts) > 0:
    print("Column | Null Count")
    print("------ | -----------")
    for col_name, count in null_counts:
        print(f"{col_name:<6} | {count}")
else:
    print("No null values found in any column of dfs_master.")

CPU times: user 5 µs, sys: 1 µs, total: 6 µs
Wall time: 8.82 µs
Verifying null values in dfs_master:
No null values found in any column of dfs_master.
